In [ ]:
"""
SUPER MINIMAL VERSION - 5 Videos Only
Copy this ENTIRE cell into Colab and run
"""

# ============================================
# CONFIGURATION - UPDATE THIS PATH
# ============================================
VAL_DIR = '/content/drive/MyDrive/val/val'  # <-- CHANGE THIS

# ============================================
# INSTALL DEPENDENCIES (if needed)
# ============================================
# !pip install -q librosa opencv-python-headless

import os
import json
import numpy as np
import pandas as pd
import cv2
import librosa
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path

print("🚀 5-VIDEO MINI PIPELINE")
print("=" * 60)

# ============================================
# STEP 1: LOAD & SAMPLE 5 VIDEOS
# ============================================
print("\n[1/4] Loading metadata...")

metadata_path = os.path.join('/content/drive/MyDrive/val/', 'val_metadata.json')
if not os.path.exists(metadata_path):
    metadata_path = os.path.join(os.path.dirname('/content/drive/MyDrive/val/'), 'val_metadata.json')

with open(metadata_path, 'r') as f:
    metadata = json.load(f)

df = pd.DataFrame(metadata)
print(f"Total dataset: {len(df):,} videos")

# Sample 5 videos (one per class)
samples = []
for mod_type in ['real', 'both_modified', 'audio_modified', 'visual_modified']:
    subset = df[(df['modify_type'] == mod_type) & (df['audio_frames'] > 0)]
    if len(subset) > 0:
        samples.append(subset.iloc[0])

# Add one more random
remaining = df[~df.index.isin([s.name for s in samples]) & (df['audio_frames'] > 0)]
if len(remaining) > 0:
    samples.append(remaining.iloc[0])

mini_df = pd.DataFrame(samples[:5]).reset_index(drop=True)
print(f"✓ Selected 5 videos")
for i, row in mini_df.iterrows():
    print(f"  {i+1}. {row['modify_type']}: {row['file'].split('/')[-1][:40]}...")

# ============================================
# STEP 2: EXTRACT FEATURES
# ============================================
print("\n[2/4] Extracting features...")

def extract_features(video_path):
    """Extract 2s AV features"""
    # Video
    cap = cv2.VideoCapture(video_path)
    frames = []
    for _ in range(50):  # 2s at 25fps
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (224, 224)) / 255.0
        frames.append(frame)
    cap.release()

    while len(frames) < 50:
        frames.append(frames[-1] if frames else np.zeros((224, 224, 3)))

    video = torch.FloatTensor(np.array(frames)).permute(0, 3, 1, 2)

    # Audio
    try:
        y, sr = librosa.load(video_path, sr=16000, duration=2.0)
        if len(y) < 32000:
            y = np.pad(y, (0, 32000 - len(y)))
        mel = librosa.feature.melspectrogram(y=y[:32000], sr=16000, n_mels=128)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        audio = torch.FloatTensor(mel_db).unsqueeze(0)
    except:
        audio = torch.zeros(1, 128, 87)

    return video, audio

features = []
for idx, row in mini_df.iterrows():
    path = os.path.join(VAL_DIR, row['file'])
    if os.path.exists(path):
        v, a = extract_features(path)

        # Labels
        mt = row['modify_type']
        if mt == 'real':
            labels = [1, 1]
        elif mt == 'both_modified':
            labels = [0, 0]
        elif mt == 'audio_modified':
            labels = [0, 1]
        else:
            labels = [1, 0]

        features.append({
            'video': v, 'audio': a,
            'labels': torch.FloatTensor(labels),
            'type': mt, 'file': row['file']
        })
        print(f"  ✓ {idx+1}/5 extracted")

print(f"✓ Features ready: {len(features)} videos")

# ============================================
# STEP 3: SIMPLE MODEL
# ============================================
print("\n[3/4] Training model...")

class MiniModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.video_enc = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(), nn.Linear(32, 64)
        )
        self.audio_enc = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(), nn.Linear(32, 64)
        )
        self.fusion = nn.Linear(128, 32)
        self.audio_head = nn.Linear(32, 1)
        self.video_head = nn.Linear(32, 1)

    def forward(self, v, a):
        # v: (B, T, C, H, W)
        b, t = v.shape[:2]
        v_feats = torch.stack([self.video_enc(v[:, i]) for i in range(t)], dim=1).mean(dim=1)
        a_feats = self.audio_enc(a)
        fused = torch.relu(self.fusion(torch.cat([v_feats, a_feats], dim=1)))
        return self.audio_head(fused), self.video_head(fused)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = MiniModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

# Train on first 3, validate on last 2
train = features[:3]
val = features[3:]

for epoch in range(10):
    model.train()
    loss_sum = 0
    for item in train:
        v = item['video'].unsqueeze(0).to(device)
        a = item['audio'].unsqueeze(0).to(device)
        l = item['labels'].unsqueeze(0).to(device)

        optimizer.zero_grad()
        a_pred, v_pred = model(v, a)
        loss = criterion(a_pred.squeeze(), l[:, 0]) + criterion(v_pred.squeeze(), l[:, 1])
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()

    if epoch % 2 == 0:
        print(f"  Epoch {epoch}: Loss = {loss_sum/len(train):.4f}")

print("✓ Training complete")

# ============================================
# STEP 4: EVALUATE
# ============================================
print("\n[4/4] Evaluation...")

model.eval()
results = []
with torch.no_grad():
    for item in features:
        v = item['video'].unsqueeze(0).to(device)
        a = item['audio'].unsqueeze(0).to(device)
        a_pred, v_pred = model(v, a)

        results.append({
            'Type': item['type'],
            'Audio_True': item['labels'][0].item(),
            'Video_True': item['labels'][1].item(),
            'Audio_Pred': torch.sigmoid(a_pred).cpu().item(),
            'Video_Pred': torch.sigmoid(v_pred).cpu().item(),
        })

results_df = pd.DataFrame(results)
print("\nResults:")
print(results_df.round(3).to_string())

# Plot
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
x = range(len(results_df))

axes[0].bar(x, results_df['Audio_True'], alpha=0.5, label='True', width=0.35)
axes[0].bar([i+0.35 for i in x], results_df['Audio_Pred'], alpha=0.5, label='Pred', width=0.35)
axes[0].set_title('Audio Predictions')
axes[0].set_ylim([0, 1.2])
axes[0].legend()

axes[1].bar(x, results_df['Video_True'], alpha=0.5, label='True', width=0.35)
axes[1].bar([i+0.35 for i in x], results_df['Video_Pred'], alpha=0.5, label='Pred', width=0.35)
axes[1].set_title('Video Predictions')
axes[1].set_ylim([0, 1.2])
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n✅ MINI PIPELINE COMPLETE!")
print("This was a test with 5 videos. Scale up to full dataset for real results.")

🚀 5-VIDEO MINI PIPELINE

[1/4] Loading metadata...
Total dataset: 77,326 videos
✓ Selected 5 videos
  1. real: real.mp4...
  2. both_modified: fake_video_fake_audio.mp4...
  3. audio_modified: real_video_fake_audio.mp4...
  4. visual_modified: fake_video_real_audio.mp4...
  5. both_modified: fake_video_fake_audio.mp4...

[2/4] Extracting features...


/tmp/ipython-input-174/318836622.py:88: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(video_path, sr=16000, duration=2.0)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


  ✓ 1/5 extracted


/tmp/ipython-input-174/318836622.py:88: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(video_path, sr=16000, duration=2.0)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


  ✓ 2/5 extracted


...

/tmp/ipython-input-174/318836622.py:88: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(video_path, sr=16000, duration=2.0)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


  ✓ 4/5 extracted


/tmp/ipython-input-174/318836622.py:88: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(video_path, sr=16000, duration=2.0)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


  ✓ 5/5 extracted
✓ Features ready: 5 videos

[3/4] Training model...


ValueError: Target size (torch.Size([1])) must be the same as input size (torch.Size([]))

In [ ]:
"""
FIXED 5-VIDEO MINI PIPELINE - Better path handling
"""

# ============================================
# CONFIGURATION - UPDATE THIS PATH
# ============================================
VAL_DIR = '/content/drive/MyDrive/AVDeepfake1M/val/'  # <-- CHANGE THIS

import os
import json
import numpy as np
import pandas as pd
import cv2
import librosa
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

print("🚀 5-VIDEO MINI PIPELINE (FIXED)")
print("=" * 60)

# ============================================
# DEBUG: Check paths
# ============================================
print("\n[DEBUG] Checking paths...")
print(f"VAL_DIR = {VAL_DIR}")
print(f"Exists: {os.path.exists(VAL_DIR)}")

if not os.path.exists(VAL_DIR):
    raise FileNotFoundError(f"Directory not found: {VAL_DIR}")

print(f"Contents: {os.listdir(VAL_DIR)[:10]}")

# ============================================
# STEP 1: LOAD METADATA
# ============================================
print("\n[1/4] Loading metadata...")

metadata_paths = [
    os.path.join(VAL_DIR, 'val_metadata.json'),
    os.path.join(VAL_DIR, '..', 'val_metadata.json'),
    os.path.join(os.path.dirname(VAL_DIR), 'val_metadata.json'),
]

metadata_path = None
for path in metadata_paths:
    if os.path.exists(path):
        metadata_path = path
        print(f"✓ Found metadata at: {path}")
        break

if metadata_path is None:
    raise FileNotFoundError("Could not find val_metadata.json")

with open(metadata_path, 'r') as f:
    metadata = json.load(f)

df = pd.DataFrame(metadata)
print(f"Total dataset: {len(df):,} videos")

# Sample 5 videos
samples = []
for mod_type in ['real', 'both_modified', 'audio_modified', 'visual_modified']:
    subset = df[(df['modify_type'] == mod_type) & (df['audio_frames'] > 0)]
    if len(subset) > 0:
        samples.append(subset.iloc[0])

while len(samples) < 5:
    remaining = df[~df.index.isin([s.name for s in samples]) & (df['audio_frames'] > 0)]
    if len(remaining) > 0:
        samples.append(remaining.iloc[0])
    else:
        break

mini_df = pd.DataFrame(samples[:5]).reset_index(drop=True)
print(f"✓ Selected {len(mini_df)} videos:")
for i, row in mini_df.iterrows():
    print(f"  {i+1}. {row['modify_type']}: {row['file']}")

# ============================================
# STEP 2: EXTRACT FEATURES
# ============================================
print("\n[2/4] Extracting features...")

def extract_features(video_path):
    if not os.path.exists(video_path):
        print(f"    ✗ File not found: {video_path}")
        return None, None

    print(f"    Processing: {os.path.basename(video_path)}")

    # Video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"    ✗ Cannot open video")
        return None, None

    frames = []
    for _ in range(50):
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (224, 224)) / 255.0
        frames.append(frame)
    cap.release()

    if len(frames) == 0:
        print(f"    ✗ No frames extracted")
        return None, None

    while len(frames) < 50:
        frames.append(frames[-1])

    video = torch.FloatTensor(np.array(frames)).permute(0, 3, 1, 2)

    # Audio
    try:
        y, sr = librosa.load(video_path, sr=16000, duration=2.0)
        if len(y) < 32000:
            y = np.pad(y, (0, 32000 - len(y)))
        mel = librosa.feature.melspectrogram(y=y[:32000], sr=16000, n_mels=128)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        audio = torch.FloatTensor(mel_db).unsqueeze(0)
    except Exception as e:
        print(f"    ⚠ Audio failed: {e}")
        audio = torch.zeros(1, 128, 87)

    return video, audio

features = []
for idx, row in mini_df.iterrows():
    # Try multiple path constructions
    possible_paths = [
        os.path.join(VAL_DIR, row['file']),
        os.path.join(VAL_DIR, row['file'].replace('val/', '')),
        os.path.join(os.path.dirname(VAL_DIR), row['file']),
    ]

    video_tensor = None
    audio_tensor = None
    used_path = None

    for path in possible_paths:
        if os.path.exists(path):
            video_tensor, audio_tensor = extract_features(path)
            if video_tensor is not None:
                used_path = path
                break

    if video_tensor is None:
        print(f"  ✗ {idx+1}/5: Could not find video")
        for p in possible_paths:
            print(f"      - {p} (exists: {os.path.exists(p)})")
        continue

    # Labels
    mt = row['modify_type']
    if mt == 'real':
        labels = [1, 1]
    elif mt == 'both_modified':
        labels = [0, 0]
    elif mt == 'audio_modified':
        labels = [0, 1]
    else:
        labels = [1, 0]

    features.append({
        'video': video_tensor,
        'audio': audio_tensor,
        'labels': torch.FloatTensor(labels),
        'type': mt,
        'file': row['file'],
        'path': used_path
    })
    print(f"  ✓ {idx+1}/5: {mt}")

print(f"\n✓ Features ready: {len(features)}/5 videos")

if len(features) == 0:
    raise RuntimeError("No videos extracted. Check paths above.")

# ============================================
# STEP 3: TRAIN MODEL
# ============================================
print("\n[3/4] Training model...")

class MiniModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.video_enc = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(), nn.Linear(32, 64)
        )
        self.audio_enc = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(), nn.Linear(32, 64)
        )
        self.fusion = nn.Linear(128, 32)
        self.audio_head = nn.Linear(32, 1)
        self.video_head = nn.Linear(32, 1)

    def forward(self, v, a):
        b, t = v.shape[:2]
        v_feats = torch.stack([self.video_enc(v[:, i]) for i in range(t)], dim=1).mean(dim=1)
        a_feats = self.audio_enc(a)
        fused = torch.relu(self.fusion(torch.cat([v_feats, a_feats], dim=1)))
        return self.audio_head(fused), self.video_head(fused)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

model = MiniModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

train = features[:3]
val = features[3:]

print(f"Training on: {len(train)} videos")
print(f"Validating on: {len(val)} videos")

for epoch in range(10):
    model.train()
    loss_sum = 0
    for item in train:
        v = item['video'].unsqueeze(0).to(device)
        a = item['audio'].unsqueeze(0).to(device)
        l = item['labels'].unsqueeze(0).to(device)

        optimizer.zero_grad()
        a_pred, v_pred = model(v, a)
        loss = criterion(a_pred.squeeze(), l[:, 0]) + criterion(v_pred.squeeze(), l[:, 1])
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()

    if len(train) > 0 and epoch % 2 == 0:
        print(f"  Epoch {epoch}: Loss = {loss_sum/len(train):.4f}")

print("✓ Training complete")

# ============================================
# STEP 4: EVALUATE
# ============================================
print("\n[4/4] Evaluation...")

model.eval()
results = []
with torch.no_grad():
    for item in features:
        v = item['video'].unsqueeze(0).to(device)
        a = item['audio'].unsqueeze(0).to(device)
        a_pred, v_pred = model(v, a)

        results.append({
            'Type': item['type'],
            'Audio_True': item['labels'][0].item(),
            'Video_True': item['labels'][1].item(),
            'Audio_Pred': torch.sigmoid(a_pred).cpu().item(),
            'Video_Pred': torch.sigmoid(v_pred).cpu().item(),
        })

results_df = pd.DataFrame(results)
print("\nResults:")
print(results_df.round(3).to_string())

# Plot
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
x = range(len(results_df))

axes[0].bar(x, results_df['Audio_True'], alpha=0.5, label='True', width=0.35)
axes[0].bar([i+0.35 for i in x], results_df['Audio_Pred'], alpha=0.5, label='Pred', width=0.35)
axes[0].set_title('Audio Predictions')
axes[0].set_ylim([0, 1.2])
axes[0].legend()

axes[1].bar(x, results_df['Video_True'], alpha=0.5, label='True', width=0.35)
axes[1].bar([i+0.35 for i in x], results_df['Video_Pred'], alpha=0.5, label='Pred', width=0.35)
axes[1].set_title('Video Predictions')
axes[1].set_ylim([0, 1.2])
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print(f"✅ COMPLETE! Processed {len(features)}/5 videos")
print("=" * 60)

🚀 5-VIDEO MINI PIPELINE (FIXED)

[DEBUG] Checking paths...
VAL_DIR = /content/drive/MyDrive/AVDeepfake1M/val/val
Exists: False


FileNotFoundError: Directory not found: /content/drive/MyDrive/AVDeepfake1M/val/val

In [ ]:
"""
COMPLETE AVDEEPFAKE1M++ PIPELINE WITH WEIGHTS & BIASES
Production-ready version with proper logging and evaluation
"""

import os
import json
import pickle
import random
import numpy as np
import pandas as pd
import cv2
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import wandb

# Set seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# =============================================================================
# CONFIGURATION
# =============================================================================

VAL_DIR = '/content/drive/MyDrive/val/extracted_val'
CONFIG = {
    "project_name": "av-deepfake-detection",
    "run_name": "avdeepfake1m-5sample-baseline",
    "architecture": "CrossModalFusion",
    "dataset": "AVDeepfake1M++",
    "samples": 5,
    "feature_dim": 128,
    "hidden_dim": 256,
    "dropout": 0.3,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "batch_size": 2,
    "epochs": 20,
    "patience": 15,
    "grad_clip": 1.0,
    "label_smoothing": 0.05
}

# =============================================================================
# SECTION 1-3: Data Loading (Your existing code)
# =============================================================================

print("=" * 60)
print("SECTION 1-3: Data Loading & Feature Extraction")
print("=" * 60)

"""
SECTION 1: Reading the JSON Metadata File
"""

import os
import json
import pandas as pd

# ============================================
# CONFIGURATION - UPDATE THIS PATH
# ============================================
VAL_DIR = '/content/drive/MyDrive/val/extracted_val'

print("=" * 60)
print("SECTION 1: Reading JSON Metadata")
print("=" * 60)

# Find metadata file
metadata_paths = [
    os.path.join(VAL_DIR, 'val_metadata.json'),
    os.path.join(VAL_DIR, '..', 'val_metadata.json'),
    os.path.join(os.path.dirname(VAL_DIR), 'val_metadata.json'),
]

metadata_path = None
for path in metadata_paths:
    if os.path.exists(path):
        metadata_path = path
        print(f"\n✓ Found metadata at: {path}")
        break

if metadata_path is None:
    raise FileNotFoundError("Could not find val_metadata.json")

# Load metadata
with open(metadata_path, 'r') as f:
    metadata = json.load(f)

print(f"✓ Loaded {len(metadata):,} entries")

# Convert to DataFrame
df = pd.DataFrame(metadata)

print(f"\nDataset Info:")
print(f"  Columns: {df.columns.tolist()}")
print(f"  Shape: {df.shape}")

print(f"\nFirst 3 entries:")
print(df.head(3)[['file', 'modify_type', 'video_frames', 'audio_frames']].to_string())

print(f"\nModification types:")
print(df['modify_type'].value_counts())

print("\n" + "=" * 60)
print("✅ Section 1 Complete - Metadata loaded")
print("=" * 60)

"""
SECTION 2: Sampling 5 Representative Videos
"""

import random
import numpy as np

print("=" * 60)
print("SECTION 2: Sampling 5 Videos")
print("=" * 60)

# Set seed for reproducibility
random.seed(42)
np.random.seed(42)

# Filter out silent videos (we need audio for AV analysis)
df_with_audio = df[df['audio_frames'] > 0].copy()
print(f"\nVideos with audio: {len(df_with_audio):,} (removed {len(df) - len(df_with_audio):,} silent)")

# Sample 1 video from each modification type
samples = []
for mod_type in ['real', 'both_modified', 'audio_modified', 'visual_modified']:
    subset = df_with_audio[df_with_audio['modify_type'] == mod_type]
    if len(subset) > 0:
        # Pick a random one
        sample = subset.sample(1, random_state=42).iloc[0]
        samples.append(sample)
        print(f"\n✓ {mod_type:20s}: {sample['file']}")

# Add one more random video (can be any type)
remaining = df_with_audio[~df_with_audio.index.isin([s.name for s in samples])]
if len(remaining) > 0:
    extra = remaining.sample(1, random_state=42).iloc[0]
    samples.append(extra)
    print(f"\n✓ {extra['modify_type']:20s}: {extra['file']} (extra)")

# Create mini dataframe
mini_df = pd.DataFrame(samples).reset_index(drop=True)

print(f"\n" + "=" * 60)
print("Selected 5 Videos Summary:")
print("=" * 60)

for i, row in mini_df.iterrows():
    print(f"\n{i+1}. {row['modify_type'].upper()}")
    print(f"   File: {row['file']}")
    print(f"   Speaker: {row['file'].split('/')[1]}")
    print(f"   Duration: {row['video_frames']/25:.1f}s ({row['video_frames']} frames)")
    print(f"   Audio: {row['audio_frames']} samples")
    print(f"   Fake segments: {len(row['fake_segments']) if isinstance(row['fake_segments'], list) else 0}")

# Save for next section
import pickle
with open('/tmp/mini_df.pkl', 'wb') as f:
    pickle.dump(mini_df, f)

print("\n" + "=" * 60)
print("✅ Section 2 Complete - 5 videos selected")
print("=" * 60)

"""
SECTION 3: Audio-Visual Feature Extraction
Extracts 2-second synchronized clips from each video
"""

import cv2
import librosa
import torch
import numpy as np
from tqdm.auto import tqdm

print("=" * 60)
print("SECTION 3: Feature Extraction")
print("=" * 60)

# Configuration
CFG = {
    'sr': 16000,           # Audio sample rate
    'fps': 25,             # Video FPS
    'duration': 2.0,       # Clip duration in seconds
    'num_frames': 50,        # 2s * 25fps
    'img_size': 224,       # Video frame size
    'audio_samples': 32000 # 2s * 16000Hz
}

def extract_av_features(video_path, fake_segments=None, total_frames=0):
    """
    Extract synchronized audio and video features

    Args:
        video_path: path to video file
        fake_segments: list of (start, end) tuples for fake parts
        total_frames: total video frames (to calculate duration)

    Returns:
        video_tensor: (50, 3, 224, 224)
        audio_tensor: (1, 128, 87) - mel spectrogram
    """

    # Determine which 2-second window to extract
    if fake_segments and len(fake_segments) > 0:
        # For fake videos: target the fake segment
        start_sec = fake_segments[0][0]
    else:
        # For real videos: take center 2 seconds
        total_sec = total_frames / CFG['fps']
        start_sec = max(0, (total_sec / 2) - (CFG['duration'] / 2))

    print(f"    Extracting from t={start_sec:.2f}s to t={start_sec+CFG['duration']:.2f}s")

    # ========== VIDEO EXTRACTION ==========
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"    ✗ Cannot open video: {video_path}")
        return None, None

    # Seek to start position
    start_frame = int(start_sec * CFG['fps'])
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    frames = []
    for _ in range(CFG['num_frames']):
        ret, frame = cap.read()
        if not ret:
            break

        # Preprocess: BGR -> RGB, resize, normalize
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (CFG['img_size'], CFG['img_size']))
        frame = frame / 255.0  # Normalize to [0, 1]
        frames.append(frame)

    cap.release()

    # Pad if video is shorter than 2 seconds
    if len(frames) < CFG['num_frames']:
        print(f"    ⚠ Video short ({len(frames)} frames), padding with last frame")
        while len(frames) < CFG['num_frames']:
            frames.append(frames[-1] if frames else np.zeros((224, 224, 3)))

    # Convert to tensor: (T, H, W, C) -> (T, C, H, W)
    video_tensor = torch.FloatTensor(np.array(frames)).permute(0, 3, 1, 2)

    # ========== AUDIO EXTRACTION ==========
    try:
        # Load audio segment
        y, sr = librosa.load(
            video_path,
            sr=CFG['sr'],
            offset=start_sec,
            duration=CFG['duration']
        )

        # Pad or truncate to exact length
        if len(y) < CFG['audio_samples']:
            y = np.pad(y, (0, CFG['audio_samples'] - len(y)))
        else:
            y = y[:CFG['audio_samples']]

        # Compute mel spectrogram
        mel_spec = librosa.feature.melspectrogram(
            y=y,
            sr=CFG['sr'],
            n_mels=128,
            n_fft=2048,
            hop_length=512
        )

        # Convert to log scale (dB)
        mel_db = librosa.power_to_db(mel_spec, ref=np.max)

        # Add channel dimension: (128, T) -> (1, 128, T)
        audio_tensor = torch.FloatTensor(mel_db).unsqueeze(0)

    except Exception as e:
        print(f"    ⚠ Audio extraction failed: {e}")
        # Return zeros as fallback
        audio_tensor = torch.zeros(1, 128, 87)

    return video_tensor, audio_tensor

# ============================================
# EXTRACT FEATURES FOR ALL 5 VIDEOS
# ============================================

print(f"\nExtracting features from {len(mini_df)} videos...")
print("-" * 60)

features = []

for idx, row in mini_df.iterrows():
    print(f"\n[{idx+1}/5] {row['modify_type'].upper()}")

    # Construct full path
    video_path = os.path.join(VAL_DIR, row['file'])

    if not os.path.exists(video_path):
        print(f"    ✗ File not found: {video_path}")
        continue

    # Extract features
    video_tensor, audio_tensor = extract_av_features(
        video_path=video_path,
        fake_segments=row.get('fake_segments'),
        total_frames=row['video_frames']
    )

    if video_tensor is None:
        print(f"    ✗ Feature extraction failed")
        continue

    # Create labels based on modification type
    mt = row['modify_type']
    if mt == 'real':
        audio_label, video_label = 1, 1  # Both real
    elif mt == 'both_modified':
        audio_label, video_label = 0, 0    # Both fake
    elif mt == 'audio_modified':
        audio_label, video_label = 0, 1    # Audio fake, video real
    else:  # visual_modified
        audio_label, video_label = 1, 0    # Audio real, video fake

    # Store everything
    features.append({
        'video': video_tensor,           # (50, 3, 224, 224)
        'audio': audio_tensor,           # (1, 128, 87)
        'labels': torch.FloatTensor([audio_label, video_label]),  # (2,)
        'type': mt,
        'file': row['file'],
        'speaker': row['file'].split('/')[1]
    })

    print(f"    ✓ Video: {video_tensor.shape}")
    print(f"    ✓ Audio: {audio_tensor.shape}")
    print(f"    ✓ Labels: Audio={audio_label}, Video={video_label}")

# ============================================
# SUMMARY
# ============================================

print(f"\n" + "=" * 60)
print("Feature Extraction Summary")
print("=" * 60)

print(f"\nSuccessfully extracted: {len(features)}/5 videos")

for i, feat in enumerate(features):
    print(f"\n{i+1}. {feat['type'].upper()}")
    print(f"   Video tensor: {feat['video'].shape}")
    print(f"   Audio tensor: {feat['audio'].shape}")
    print(f"   Labels: [Audio={feat['labels'][0]:.0f}, Video={feat['labels'][1]:.0f}]")

# Save for next section
with open('/tmp/features.pkl', 'wb') as f:
    pickle.dump(features, f)

print("\n" + "=" * 60)
print("✅ Section 3 Complete - Features extracted")
print("=" * 60)

# For this example, recreate minimal dataset if needed
if 'features' not in globals():
    print("Loading features from /tmp/features.pkl...")
    try:
        with open('/tmp/features.pkl', 'rb') as f:
            features = pickle.load(f)
    except:
        raise FileNotFoundError("Please run Sections 1-3 first to extract features")

print(f"Loaded {len(features)} samples")

# =============================================================================
# SECTION 4: Improved Model Architecture
# =============================================================================

print("=" * 60)
print("SECTION 4: Improved Cross-Modal Model")
print("=" * 60)

class ImprovedAVDeepfakeDetector(nn.Module):
    """
    Production-ready AV Deepfake Detector with:
    - Batch normalization for stable training
    - Residual-style connections
    - Proper regularization
    - Temperature-scaled predictions
    """

    def __init__(self, feature_dim=128, hidden_dim=256, dropout=0.3):
        super().__init__()

        # Video Encoder (3D CNN)
        self.video_conv1 = nn.Sequential(
            nn.Conv3d(3, 32, kernel_size=(3, 7, 7), stride=(1, 2, 2), padding=(1, 3, 3)),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.MaxPool3d((1, 2, 2))
        )

        self.video_conv2 = nn.Sequential(
            nn.Conv3d(32, 64, kernel_size=(3, 5, 5), stride=(1, 2, 2), padding=(1, 2, 2)),
            nn.BatchNorm3d(64),
            nn.ReLU(),
            nn.MaxPool3d((2, 2, 2))
        )

        self.video_conv3 = nn.Sequential(
            nn.Conv3d(64, 128, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool3d((1, 4, 4))
        )

        self.video_fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, feature_dim),
            nn.Dropout(dropout),
            nn.ReLU()
        )

        # Audio Encoder (2D CNN)
        self.audio_conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )

        self.audio_conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.audio_conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.audio_fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, feature_dim),
            nn.Dropout(dropout),
            nn.ReLU()
        )

        # Cross-modal Fusion
        self.fusion = nn.Sequential(
            nn.Linear(feature_dim * 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout/2)
        )

        # Classifiers
        self.audio_classifier = nn.Linear(hidden_dim, 1)
        self.video_classifier = nn.Linear(hidden_dim, 1)
        self.joint_classifier = nn.Linear(hidden_dim, 1)

        self.temperature = 0.1  # Temperature for sigmoid scaling

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv3d, nn.Conv2d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, (nn.BatchNorm3d, nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, video, audio):
        # Video: (B, T, C, H, W) -> (B, C, T, H, W)
        video = video.permute(0, 2, 1, 3, 4)

        # Encode video
        v = self.video_conv1(video)
        v = self.video_conv2(v)
        v = self.video_conv3(v)
        video_feat = self.video_fc(v)

        # Encode audio
        a = self.audio_conv1(audio)
        a = self.audio_conv2(a)
        a = self.audio_conv3(a)
        audio_feat = self.audio_fc(a)

        # Fusion
        combined = torch.cat([video_feat, audio_feat], dim=1)
        fused = self.fusion(combined)

        # Predictions with temperature scaling
        audio_pred = torch.sigmoid(self.audio_classifier(fused) / self.temperature)
        video_pred = torch.sigmoid(self.video_classifier(fused) / self.temperature)
        joint_pred = torch.sigmoid(self.joint_classifier(fused) / self.temperature)

        return {
            'audio_pred': audio_pred,
            'video_pred': video_pred,
            'joint_pred': joint_pred,
            'features': {
                'video': video_feat,
                'audio': audio_feat,
                'fused': fused
            }
        }

# Initialize model
model = ImprovedAVDeepfakeDetector(
    feature_dim=CONFIG['feature_dim'],
    hidden_dim=CONFIG['hidden_dim'],
    dropout=CONFIG['dropout']
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model initialized:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size: {total_params * 4 / 1024 / 1024:.2f} MB")

# Test forward pass
with torch.no_grad():
    test_video = torch.randn(2, 50, 3, 224, 224).to(device)
    test_audio = torch.randn(2, 1, 128, 87).to(device)
    test_out = model(test_video, test_audio)
    print(f"  Output shapes: Audio {test_out['audio_pred'].shape}, "
          f"Video {test_out['video_pred'].shape}, Joint {test_out['joint_pred'].shape}")

# =============================================================================
# SECTION 5: W&B Setup & Training
# =============================================================================

print("=" * 60)
print("SECTION 5: W&B Initialization & Training")
print("=" * 60)

# Initialize W&B
wandb.init(
    project=CONFIG['project_name'],
    name=CONFIG['run_name'],
    config=CONFIG,
    tags=["baseline", "5-samples", "cross-modal"],
    notes="Initial baseline with 5 samples from AVDeepfake1M++"
)

# Log model architecture
wandb.watch(model, log="all", log_freq=10)

# Dataset
class AVDataset(torch.utils.data.Dataset):
    def __init__(self, features):
        self.features = features

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        item = self.features[idx]
        return {
            'video': item['video'],
            'audio': item['audio'],
            'labels': item['labels'],
            'type': item['type'],
            'file': item['file']
        }

dataset = AVDataset(features)
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=0
)

# Optimizer & Scheduler
optimizer = optim.AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5,
    # verbose=True
)

# Loss functions
class LabelSmoothingBCE(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.confidence = 1.0 - smoothing

    def forward(self, pred, target):
        target = target * self.confidence + 0.5 * self.smoothing
        return F.binary_cross_entropy(pred, target)

criterion = LabelSmoothingBCE(smoothing=CONFIG['label_smoothing'])
criterion_hard = nn.BCELoss()

# Training loop
print(f"\nStarting training for max {CONFIG['epochs']} epochs...")
print(f"Early stopping patience: {CONFIG['patience']}")

best_val_loss = float('inf')
patience_counter = 0
global_step = 0

# History for plotting
history = {
    'train_loss': [], 'val_loss': [],
    'train_auc': [], 'val_auc': [],
    'audio_auc': [], 'video_auc': [], 'joint_auc': [],
    'learning_rate': []
}

for epoch in range(CONFIG['epochs']):
    # Training
    model.train()
    epoch_loss = 0
    train_preds = {'audio': [], 'video': [], 'joint': []}
    train_labels = {'audio': [], 'video': [], 'joint': []}

    # Multiple passes over small dataset
    num_repeats = 10

    for repeat in range(num_repeats):
        for batch in dataloader:
            video = batch['video'].to(device)
            audio = batch['audio'].to(device)
            labels = batch['labels'].to(device)

            audio_labels = labels[:, 0:1]
            video_labels = labels[:, 1:2]
            joint_labels = ((audio_labels == 1) & (video_labels == 1)).float()

            optimizer.zero_grad()
            outputs = model(video, audio)

            # Multi-task loss
            loss_audio = criterion(outputs['audio_pred'], audio_labels)
            loss_video = criterion(outputs['video_pred'], video_labels)
            loss_joint = criterion_hard(outputs['joint_pred'], joint_labels)

            loss = loss_audio + loss_video + 2.0 * loss_joint

            loss.backward()

            # Gradient clipping and logging
            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                CONFIG['grad_clip']
            )

            optimizer.step()

            epoch_loss += loss.item()

            # Store predictions
            train_preds['audio'].extend(outputs['audio_pred'].detach().cpu().numpy())
            train_preds['video'].extend(outputs['video_pred'].detach().cpu().numpy())
            train_preds['joint'].extend(outputs['joint_pred'].detach().cpu().numpy())
            train_labels['audio'].extend(audio_labels.cpu().numpy())
            train_labels['video'].extend(video_labels.cpu().numpy())
            train_labels['joint'].extend(joint_labels.cpu().numpy())

            global_step += 1

            # Log step-level metrics every 10 steps
            if global_step % 10 == 0:
                wandb.log({
                    "train/step_loss": loss.item(),
                    "train/grad_norm": grad_norm.item(),
                    "train/learning_rate": optimizer.param_groups[0]['lr'],
                    "step": global_step
                })

    # Calculate training metrics
    avg_train_loss = epoch_loss / (len(dataloader) * num_repeats)

    def calc_auc(y_true, y_pred):
        y_true = np.array(y_true).flatten()
        y_pred = np.array(y_pred).flatten()
        if len(np.unique(y_true)) < 2:
            return 0.5
        try:
            return roc_auc_score(y_true, y_pred)
        except:
            return 0.5

    train_auc_audio = calc_auc(train_labels['audio'], train_preds['audio'])
    train_auc_video = calc_auc(train_labels['video'], train_preds['video'])
    train_auc_joint = calc_auc(train_labels['joint'], train_preds['joint'])

    # Validation
    model.eval()
    val_loss = 0
    val_preds = {'audio': [], 'video': [], 'joint': []}
    val_labels = {'audio': [], 'video': [], 'joint': []}
    val_types = []

    with torch.no_grad():
        for batch in dataloader:
            video = batch['video'].to(device)
            audio = batch['audio'].to(device)
            labels = batch['labels'].to(device)

            audio_labels = labels[:, 0:1]
            video_labels = labels[:, 1:2]
            joint_labels = ((audio_labels == 1) & (video_labels == 1)).float()

            outputs = model(video, audio)

            loss = (criterion_hard(outputs['audio_pred'], audio_labels) +
                   criterion_hard(outputs['video_pred'], video_labels) +
                   2.0 * criterion_hard(outputs['joint_pred'], joint_labels))

            val_loss += loss.item()

            val_preds['audio'].extend(outputs['audio_pred'].cpu().numpy())
            val_preds['video'].extend(outputs['video_pred'].cpu().numpy())
            val_preds['joint'].extend(outputs['joint_pred'].cpu().numpy())
            val_labels['audio'].extend(audio_labels.cpu().numpy())
            val_labels['video'].extend(video_labels.cpu().numpy())
            val_labels['joint'].extend(joint_labels.cpu().numpy())
            val_types.extend(batch['type'])

    avg_val_loss = val_loss / len(dataloader)
    val_auc_audio = calc_auc(val_labels['audio'], val_preds['audio'])
    val_auc_video = calc_auc(val_labels['video'], val_preds['video'])
    val_auc_joint = calc_auc(val_labels['joint'], val_preds['joint'])

    # Update history
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_auc'].append(train_auc_joint)
    history['val_auc'].append(val_auc_joint)
    history['audio_auc'].append(val_auc_audio)
    history['video_auc'].append(val_auc_video)
    history['joint_auc'].append(val_auc_joint)
    history['learning_rate'].append(optimizer.param_groups[0]['lr'])

    # Log epoch metrics to W&B
    log_dict = {
        "epoch": epoch,
        "train/loss": avg_train_loss,
        "train/auc_audio": train_auc_audio,
        "train/auc_video": train_auc_video,
        "train/auc_joint": train_auc_joint,
        "val/loss": avg_val_loss,
        "val/auc_audio": val_auc_audio,
        "val/auc_video": val_auc_video,
        "val/auc_joint": val_auc_joint,
        "val/pred_mean_audio": np.mean(val_preds['audio']),
        "val/pred_std_audio": np.std(val_preds['audio']),
        "val/pred_mean_video": np.mean(val_preds['video']),
        "val/pred_std_video": np.std(val_preds['video']),
        "val/pred_mean_joint": np.mean(val_preds['joint']),
        "val/pred_std_joint": np.std(val_preds['joint']),
    }

    # Log prediction distributions as histograms
    if epoch % 5 == 0:
        log_dict["predictions/audio"] = wandb.Histogram(val_preds['audio'])
        log_dict["predictions/video"] = wandb.Histogram(val_preds['video'])
        log_dict["predictions/joint"] = wandb.Histogram(val_preds['joint'])

    wandb.log(log_dict)

    # Print progress
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:3d}/{CONFIG['epochs']}] "
              f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | "
              f"AUCs: A={val_auc_audio:.3f} V={val_auc_video:.3f} J={val_auc_joint:.3f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.2e}")

    # Learning rate scheduling
    prev_lr = optimizer.param_groups[0]['lr']
    scheduler.step(avg_val_loss)
    new_lr = optimizer.param_groups[0]['lr']
    if new_lr != prev_lr:
        print(f"  ↳ Learning rate reduced: {prev_lr:.2e} -> {new_lr:.2e}")

    # Early stopping
    if avg_val_loss < best_val_loss - 0.0001:
        best_val_loss = avg_val_loss
        patience_counter = 0

        # Save best model
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': best_val_loss,
            'auc_joint': val_auc_joint,
            'config': CONFIG
        }, '/tmp/best_av_model.pth')

        # Log to W&B
        wandb.save('/tmp/best_av_model.pth')

        print(f"  ✓ New best model saved (loss: {best_val_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= CONFIG['patience']:
            print(f"\n⏹️ Early stopping triggered at epoch {epoch+1}")
            break

print("\n✅ Training complete")

# =============================================================================
# SECTION 6: Visualization & Final Evaluation
# =============================================================================

print("=" * 60)
print("SECTION 6: Final Evaluation & Visualization")
print("=" * 60)

# Load best model
checkpoint = torch.load('/tmp/best_av_model.pth', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Loaded best model from epoch {checkpoint['epoch']+1}")
print(f"Best validation loss: {checkpoint['loss']:.4f}")
print(f"Best joint AUC: {checkpoint['auc_joint']:.4f}")

# Generate comprehensive plots
fig = plt.figure(figsize=(20, 12))

# Create grid for subplots
gs = fig.add_gridspec(3, 4, hspace=0.3, wspace=0.3)

# 1. Loss curves
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(history['train_loss'], label='Train', linewidth=2)
ax1.plot(history['val_loss'], label='Val', linewidth=2)
ax1.set_title('Loss Curves', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. AUC curves
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(history['audio_auc'], label='Audio', linewidth=2)
ax2.plot(history['video_auc'], label='Video', linewidth=2)
ax2.plot(history['joint_auc'], label='Joint', linewidth=2)
ax2.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Random')
ax2.set_title('Validation AUC', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('AUC')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Learning rate
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(history['learning_rate'], linewidth=2, color='green')
ax3.set_title('Learning Rate Schedule', fontsize=12, fontweight='bold')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('LR')
ax3.set_yscale('log')
ax3.grid(True, alpha=0.3)

# 4. Overfitting analysis
ax4 = fig.add_subplot(gs[0, 3])
gap = np.array(history['train_loss']) - np.array(history['val_loss'])
ax4.plot(gap, linewidth=2, color='purple')
ax4.axhline(y=0, color='r', linestyle='--', alpha=0.5)
ax4.set_title('Generalization Gap', fontsize=12, fontweight='bold')
ax4.set_xlabel('Epoch')
ax4.set_ylabel('Train Loss - Val Loss')
ax4.grid(True, alpha=0.3)

# Get final predictions for detailed analysis
final_preds = []
with torch.no_grad():
    for feat in features:
        video = feat['video'].unsqueeze(0).to(device)
        audio = feat['audio'].unsqueeze(0).to(device)

        outputs = model(video, audio)

        final_preds.append({
            'file': feat['file'],
            'type': feat['type'],
            'audio_gt': feat['labels'][0].item(),
            'video_gt': feat['labels'][1].item(),
            'audio_pred': outputs['audio_pred'].item(),
            'video_pred': outputs['video_pred'].item(),
            'joint_pred': outputs['joint_pred'].item()
        })

df_final = pd.DataFrame(final_preds)

# 5. Prediction scatter plot
ax5 = fig.add_subplot(gs[1, :2])
colors = {'real': 'green', 'audio_modified': 'blue', 'visual_modified': 'orange', 'both_modified': 'red'}
for mod_type in df_final['type'].unique():
    subset = df_final[df_final['type'] == mod_type]
    ax5.scatter(subset['audio_pred'], subset['video_pred'],
               c=colors.get(mod_type, 'gray'), label=mod_type, s=100, alpha=0.7)
ax5.plot([0, 1], [0.5, 0.5], 'k--', alpha=0.3)
ax5.plot([0.5, 0.5], [0, 1], 'k--', alpha=0.3)
ax5.set_xlabel('Audio Prediction (0=Fake, 1=Real)')
ax5.set_ylabel('Video Prediction (0=Fake, 1=Real)')
ax5.set_title('Audio vs Video Predictions by Type', fontsize=12, fontweight='bold')
ax5.legend()
ax5.grid(True, alpha=0.3)

# 6. Per-type bar chart
ax6 = fig.add_subplot(gs[1, 2:])
type_means = df_final.groupby('type')[['audio_pred', 'video_pred', 'joint_pred']].mean()
type_means.plot(kind='bar', ax=ax6, color=['skyblue', 'lightcoral', 'lightgreen'])
ax6.set_title('Average Predictions by Modification Type', fontsize=12, fontweight='bold')
ax6.set_ylabel('Prediction Score')
ax6.legend(['Audio', 'Video', 'Joint'])
ax6.grid(True, alpha=0.3)
plt.setp(ax6.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 7. Confusion matrix for joint predictions
ax7 = fig.add_subplot(gs[2, 0])
joint_gt = ((df_final['audio_gt'] == 0) | (df_final['video_gt'] == 0)).astype(int)
joint_pred_bin = (df_final['joint_pred'] > 0.5).astype(int)
cm = confusion_matrix(joint_gt, joint_pred_bin)
im = ax7.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax7.set_title('Joint Prediction Confusion Matrix', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax7)
tick_marks = np.arange(2)
ax7.set_xticks(tick_marks)
ax7.set_yticks(tick_marks)
ax7.set_xticklabels(['Fake', 'Real'])
ax7.set_yticklabels(['Fake', 'Real'])
ax7.set_ylabel('True Label')
ax7.set_xlabel('Predicted Label')
for i in range(2):
    for j in range(2):
        ax7.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center", color="black", fontsize=12)

# 8. Sample-wise results table
ax8 = fig.add_subplot(gs[2, 1:])
ax8.axis('tight')
ax8.axis('off')
table_data = []
for _, row in df_final.iterrows():
    audio_status = "✓" if ((row['audio_pred'] > 0.5) == (row['audio_gt'] == 1)) else "✗"
    video_status = "✓" if ((row['video_pred'] > 0.5) == (row['video_gt'] == 1)) else "✗"
    joint_gt = 0 if (row['audio_gt'] == 0 or row['video_gt'] == 0) else 1
    joint_status = "✓" if ((row['joint_pred'] > 0.5) == joint_gt) else "✗"

    table_data.append([
        row['type'][:15],
        f"{row['audio_pred']:.3f} {audio_status}",
        f"{row['video_pred']:.3f} {video_status}",
        f"{row['joint_pred']:.3f} {joint_status}"
    ])

table = ax8.table(cellText=table_data,
                 colLabels=['Type', 'Audio Pred', 'Video Pred', 'Joint Pred'],
                 cellLoc='center',
                 loc='center',
                 colWidths=[0.3, 0.2, 0.2, 0.2])
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2)
ax8.set_title('Sample-wise Predictions (✓=Correct, ✗=Wrong)', fontsize=12, fontweight='bold', pad=20)

plt.suptitle(f'AV Deepfake Detection Results - {CONFIG["run_name"]}',
             fontsize=16, fontweight='bold', y=0.98)

plt.savefig('/tmp/final_results.png', dpi=150, bbox_inches='tight', facecolor='white')
wandb.log({"final_results": wandb.Image('/tmp/final_results.png')})
plt.show()

# Calculate final metrics
print("\n" + "=" * 60)
print("FINAL METRICS")
print("=" * 60)

def calculate_metrics(y_true, y_pred, name):
    y_pred_bin = (y_pred > 0.5).astype(int)
    acc = accuracy_score(y_true, y_pred_bin)
    prec = precision_score(y_true, y_pred_bin, zero_division=0)
    rec = recall_score(y_true, y_pred_bin, zero_division=0)
    f1 = f1_score(y_true, y_pred_bin, zero_division=0)
    auc = roc_auc_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.5

    print(f"\n{name}:")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  AUC:       {auc:.4f}")

    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'auc': auc}

metrics_audio = calculate_metrics(df_final['audio_gt'], df_final['audio_pred'], "Audio")
metrics_video = calculate_metrics(df_final['video_gt'], df_final['video_pred'], "Video")

joint_gt = ((df_final['audio_gt'] == 0) | (df_final['video_gt'] == 0)).astype(int)
metrics_joint = calculate_metrics(joint_gt, df_final['joint_pred'], "Joint")

# Log final summary to W&B
wandb.summary.update({
    "best_epoch": checkpoint['epoch'],
    "best_val_loss": checkpoint['loss'],
    "best_val_auc_joint": checkpoint['auc_joint'],
    "final_audio_accuracy": metrics_audio['accuracy'],
    "final_audio_auc": metrics_audio['auc'],
    "final_video_accuracy": metrics_video['accuracy'],
    "final_video_auc": metrics_video['auc'],
    "final_joint_accuracy": metrics_joint['accuracy'],
    "final_joint_auc": metrics_joint['auc'],
    "total_parameters": total_params,
    "training_epochs": len(history['train_loss'])
})

# Log final predictions table
wandb.log({"final_predictions": wandb.Table(dataframe=df_final)})

print("\n" + "=" * 60)
print("✅ W&B Run Complete")
print("=" * 60)
print(f"\nView your run at: {wandb.run.url}")
print(f"Project dashboard: https://wandb.ai/{wandb.run.entity}/{CONFIG['project_name']}")

wandb.finish()

Using device: cpu
SECTION 1-3: Data Loading & Feature Extraction
SECTION 1: Reading JSON Metadata

✓ Found metadata at: /content/drive/MyDrive/val/extracted_val/val_metadata.json
✓ Loaded 77,326 entries

Dataset Info:
  Columns: ['file', 'original', 'split', 'modify_type', 'audio_model', 'fake_segments', 'audio_fake_segments', 'visual_fake_segments', 'video_frames', 'audio_frames', 'video_model']
  Shape: (77326, 11)

First 3 entries:
                                                              file    modify_type  video_frames  audio_frames
0  vox_celeb_2/id01358/_1nATum8x78/00030/fake_video_fake_audio.mp4  both_modified           161        102656
1  vox_celeb_2/id01358/_1nATum8x78/00027/fake_video_fake_audio.mp4  both_modified           166        106880
2                   vox_celeb_2/id01358/_1nATum8x78/00025/real.mp4           real           139         89088

Modification types:
modify_type
real               20220
visual_modified    19099
both_modified      19069
audio_modifie

/tmp/ipython-input-174/3793071568.py:264: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


    ✓ Video: torch.Size([50, 3, 224, 224])
    ✓ Audio: torch.Size([1, 128, 63])
    ✓ Labels: Audio=1, Video=1

[2/5] BOTH_MODIFIED
    Extracting from t=2.70s to t=4.70s


/tmp/ipython-input-174/3793071568.py:264: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


    ✓ Video: torch.Size([50, 3, 224, 224])
    ✓ Audio: torch.Size([1, 128, 63])
    ✓ Labels: Audio=0, Video=0

[3/5] AUDIO_MODIFIED
    Extracting from t=0.00s to t=2.00s


...

    ✓ Video: torch.Size([50, 3, 224, 224])
    ✓ Audio: torch.Size([1, 128, 63])
    ✓ Labels: Audio=1, Video=0

[5/5] VISUAL_MODIFIED
    Extracting from t=4.90s to t=6.90s


/tmp/ipython-input-174/3793071568.py:264: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


    ✓ Video: torch.Size([50, 3, 224, 224])
    ✓ Audio: torch.Size([1, 128, 63])
    ✓ Labels: Audio=1, Video=0

Feature Extraction Summary

Successfully extracted: 5/5 videos

1. REAL
   Video tensor: torch.Size([50, 3, 224, 224])
   Audio tensor: torch.Size([1, 128, 63])
   Labels: [Audio=1, Video=1]

2. BOTH_MODIFIED
   Video tensor: torch.Size([50, 3, 224, 224])
   Audio tensor: torch.Size([1, 128, 63])
   Labels: [Audio=0, Video=0]

3. AUDIO_MODIFIED
   Video tensor: torch.Size([50, 3, 224, 224])
   Audio tensor: torch.Size([1, 128, 63])
   Labels: [Audio=0, Video=1]

4. VISUAL_MODIFIED
   Video tensor: torch.Size([50, 3, 224, 224])
   Audio tensor: torch.Size([1, 128, 63])
   Labels: [Audio=1, Video=0]

5. VISUAL_MODIFIED
   Video tensor: torch.Size([50, 3, 224, 224])
   Audio tensor: torch.Size([1, 128, 63])
   Labels: [Audio=1, Video=0]

✅ Section 3 Complete - Features extracted
Loaded 5 samples
SECTION 4: Improved Cross-Modal Model
Model initialized:
  Total parameters: 1,140,


Starting training for max 100 epochs...
Early stopping patience: 15


ValueError: Expected more than 1 value per channel when training, got input size torch.Size([1, 256])

# Mini pipeline


In [1]:
from google.colab import drive
# Mounting the gogle drive to store the data
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
 """
COMPLETE AVDEEPFAKE1M++ PIPELINE WITH PRE-TRAINED ENCODERS
Uses original feature extraction: video (50, 3, 224, 224), audio (1, 128, 87)
"""

import os
import json
import pickle
import random
import numpy as np
import pandas as pd
import cv2
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import torchvision.models as models
import subprocess
import sys

try:
    import wandb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "wandb", "-q"])
    import wandb

# Set seeds
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# =============================================================================
# CONFIGURATION
# =============================================================================

VAL_DIR = '/content/drive/MyDrive/val/extracted_val'

CONFIG = {
    "project_name": "av-deepfake-detection",
    "run_name": "pretrained-encoders-original-features",
    "architecture": "PretrainedResNet3D_ResNet18",
    "dataset": "AVDeepfake1M++",
    "samples": 5,
    "video_encoder": "r3d_18",
    "audio_encoder": "resnet18",
    "feature_dim": 256,
    "hidden_dim": 512,
    "dropout": 0.4,
    "learning_rate": 1e-4,
    "encoder_lr": 1e-5,
    "weight_decay": 1e-4,
    "batch_size": 2,
    "epochs": 20,
    "freeze_epochs": 5,
    "patience": 10,
    "grad_clip": 1.0,
    "label_smoothing": 0.05
}

Using device: cpu


In [3]:
# =============================================================================
# SECTION 1: Reading JSON Metadata
# =============================================================================

VAL_DIR = '/content/drive/MyDrive/val/extracted_val'
print("=" * 60)
print("SECTION 1: Reading JSON Metadata")
print("=" * 60)

metadata_paths = [
    os.path.join(VAL_DIR, 'val_metadata.json'),
    os.path.join(VAL_DIR, '..', 'val_metadata.json'),
    os.path.join(os.path.dirname(VAL_DIR), 'val_metadata.json'),
]

metadata_path = None
for path in metadata_paths:
    if os.path.exists(path):
        metadata_path = path
        print(f"\n✓ Found metadata at: {path}")
        break

if metadata_path is None:
    raise FileNotFoundError("Could not find val_metadata.json")

with open(metadata_path, 'r') as f:
    metadata = json.load(f)

print(f"✓ Loaded {len(metadata):,} entries")

df = pd.DataFrame(metadata)

print(f"\nDataset Info:")
print(f"  Columns: {df.columns.tolist()}")
print(f"  Shape: {df.shape}")

print(f"\nFirst 3 entries:")
print(df.head(3)[['file', 'modify_type', 'video_frames', 'audio_frames']].to_string())

print(f"\nModification types:")
print(df['modify_type'].value_counts())

print("\n" + "=" * 60)
print("Section 1 Complete - Metadata loaded")
print("=" * 60)


SECTION 1: Reading JSON Metadata

✓ Found metadata at: /content/drive/MyDrive/val/extracted_val/val_metadata.json
✓ Loaded 77,326 entries

Dataset Info:
  Columns: ['file', 'original', 'split', 'modify_type', 'audio_model', 'fake_segments', 'audio_fake_segments', 'visual_fake_segments', 'video_frames', 'audio_frames', 'video_model']
  Shape: (77326, 11)

First 3 entries:
                                                              file    modify_type  video_frames  audio_frames
0  vox_celeb_2/id01358/_1nATum8x78/00030/fake_video_fake_audio.mp4  both_modified           161        102656
1  vox_celeb_2/id01358/_1nATum8x78/00027/fake_video_fake_audio.mp4  both_modified           166        106880
2                   vox_celeb_2/id01358/_1nATum8x78/00025/real.mp4           real           139         89088

Modification types:
modify_type
real               20220
visual_modified    19099
both_modified      19069
audio_modified     18938
Name: count, dtype: int64

Section 1 Complete - Metad

In [5]:
# =============================================================================
# SECTION 2: Sampling 5 Representative Videos
# =============================================================================

print("=" * 60)
print("SECTION 2: Sampling 5 Videos")
print("=" * 60)

random.seed(42)
np.random.seed(42)

df_with_audio = df[df['audio_frames'] > 0].copy()
print(f"\nVideos with audio: {len(df_with_audio):,} (removed {len(df) - len(df_with_audio):,} silent)")

samples = []
for mod_type in ['real', 'both_modified', 'audio_modified', 'visual_modified']:
    subset = df_with_audio[df_with_audio['modify_type'] == mod_type]
    if len(subset) > 0:
        sample = subset.sample(1, random_state=42).iloc[0]
        samples.append(sample)
        print(f"\n✓ {mod_type:20s}: {sample['file']}")

remaining = df_with_audio[~df_with_audio.index.isin([s.name for s in samples])]
if len(remaining) > 0:
    extra = remaining.sample(1, random_state=42).iloc[0]
    samples.append(extra)
    print(f"\n✓ {extra['modify_type']:20s}: {extra['file']} (extra)")

mini_df = pd.DataFrame(samples).reset_index(drop=True)

print(f"\n" + "=" * 60)
print("Selected 5 Videos Summary:")
print("=" * 60)

for i, row in mini_df.iterrows():
    print(f"\n{i+1}. {row['modify_type'].upper()}")
    print(f"   File: {row['file']}")
    print(f"   Speaker: {row['file'].split('/')[1]}")
    print(f"   Duration: {row['video_frames']/25:.1f}s ({row['video_frames']} frames)")
    print(f"   Audio: {row['audio_frames']} samples")
    print(f"   Fake segments: {len(row['fake_segments']) if isinstance(row['fake_segments'], list) else 0}")

with open('/tmp/mini_df.pkl', 'wb') as f:
    pickle.dump(mini_df, f)

print("\n" + "=" * 60)
print("✅ Section 2 Complete - 5 videos selected")
print("=" * 60)


SECTION 2: Sampling 5 Videos

Videos with audio: 77,115 (removed 211 silent)

✓ real                : vox_celeb_2/id00570/grpbnMi_NPo/00079/real_p1.mp4

✓ both_modified       : vox_celeb_2/id02700/aSUjfY7xs0w/00053/fake_video_fake_audio.mp4

✓ audio_modified      : vox_celeb_2/id08702/g_VOR_5UoqU/00258/real_video_fake_audio.mp4

✓ visual_modified     : vox_celeb_2/id03942/WxMIBMSjjlM/00329/fake_video_real_audio.mp4

✓ visual_modified     : vox_celeb_2/id08661/7CCvNxe4Fd4/00010/fake_video_real_audio.mp4 (extra)

Selected 5 Videos Summary:

1. REAL
   File: vox_celeb_2/id00570/grpbnMi_NPo/00079/real_p1.mp4
   Speaker: id00570
   Duration: 7.3s (182 frames)
   Audio: 116736 samples
   Fake segments: 0

2. BOTH_MODIFIED
   File: vox_celeb_2/id02700/aSUjfY7xs0w/00053/fake_video_fake_audio.mp4
   Speaker: id02700
   Duration: 6.6s (166 frames)
   Audio: 107200 samples
   Fake segments: 1

3. AUDIO_MODIFIED
   File: vox_celeb_2/id08702/g_VOR_5UoqU/00258/real_video_fake_audio.mp4
   Speaker: i

In [6]:
# =============================================================================
# SECTION 3: Audio-Visual Feature Extraction
# =============================================================================

print("=" * 60)
print("SECTION 3: Feature Extraction")
print("=" * 60)

CFG = {
    'sr': 16000,
    'fps': 25,
    'duration': 2.0,
    'num_frames': 50,
    'img_size': 224,
    'audio_samples': 32000
}

def extract_av_features(video_path, fake_segments=None, total_frames=0):
    if fake_segments and len(fake_segments) > 0:
        start_sec = fake_segments[0][0]
    else:
        total_sec = total_frames / CFG['fps']
        start_sec = max(0, (total_sec / 2) - (CFG['duration'] / 2))

    print(f"    Extracting from t={start_sec:.2f}s to t={start_sec+CFG['duration']:.2f}s")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"    ✗ Cannot open video: {video_path}")
        return None, None

    start_frame = int(start_sec * CFG['fps'])
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    frames = []
    for _ in range(CFG['num_frames']):
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (CFG['img_size'], CFG['img_size']))
        frame = frame / 255.0
        frames.append(frame)
    cap.release()

    if len(frames) < CFG['num_frames']:
        print(f"    ⚠ Video short ({len(frames)} frames), padding with last frame")
        while len(frames) < CFG['num_frames']:
            frames.append(frames[-1] if frames else np.zeros((224, 224, 3)))

    video_tensor = torch.FloatTensor(np.array(frames)).permute(0, 3, 1, 2)

    try:
        y, sr = librosa.load(video_path, sr=CFG['sr'], offset=start_sec, duration=CFG['duration'])
        if len(y) < CFG['audio_samples']:
            y = np.pad(y, (0, CFG['audio_samples'] - len(y)))
        else:
            y = y[:CFG['audio_samples']]

        mel_spec = librosa.feature.melspectrogram(y=y, sr=CFG['sr'], n_mels=128, n_fft=2048, hop_length=512)
        mel_db = librosa.power_to_db(mel_spec, ref=np.max)
        audio_tensor = torch.FloatTensor(mel_db).unsqueeze(0)

    except Exception as e:
        print(f"    ⚠ Audio extraction failed: {e}")
        audio_tensor = torch.zeros(1, 128, 87)

    return video_tensor, audio_tensor

print(f"\nExtracting features from {len(mini_df)} videos...")
print("-" * 60)

features = []

for idx, row in mini_df.iterrows():
    print(f"\n[{idx+1}/5] {row['modify_type'].upper()}")
    video_path = os.path.join(VAL_DIR, row['file'])

    if not os.path.exists(video_path):
        print(f"    ✗ File not found: {video_path}")
        continue

    video_tensor, audio_tensor = extract_av_features(
        video_path=video_path,
        fake_segments=row.get('fake_segments'),
        total_frames=row['video_frames']
    )

    if video_tensor is None:
        print(f"    ✗ Feature extraction failed")
        continue

    mt = row['modify_type']
    if mt == 'real':
        audio_label, video_label = 1, 1
    elif mt == 'both_modified':
        audio_label, video_label = 0, 0
    elif mt == 'audio_modified':
        audio_label, video_label = 0, 1
    else:
        audio_label, video_label = 1, 0

    features.append({
        'video': video_tensor,
        'audio': audio_tensor,
        'labels': torch.FloatTensor([audio_label, video_label]),
        'type': mt,
        'file': row['file'],
        'speaker': row['file'].split('/')[1]
    })

    print(f"    ✓ Video: {video_tensor.shape}")
    print(f"    ✓ Audio: {audio_tensor.shape}")
    print(f"    ✓ Labels: Audio={audio_label}, Video={video_label}")

print(f"\n" + "=" * 60)
print("Feature Extraction Summary")
print("=" * 60)

print(f"\nSuccessfully extracted: {len(features)}/5 videos")

for i, feat in enumerate(features):
    print(f"\n{i+1}. {feat['type'].upper()}")
    print(f"   Video tensor: {feat['video'].shape}")
    print(f"   Audio tensor: {feat['audio'].shape}")
    print(f"   Labels: [Audio={feat['labels'][0]:.0f}, Video={feat['labels'][1]:.0f}]")

with open('/tmp/features.pkl', 'wb') as f:
    pickle.dump(features, f)

print("\n" + "=" * 60)
print("✅ Section 3 Complete - Features extracted")
print("=" * 60)

SECTION 3: Feature Extraction

Extracting features from 5 videos...
------------------------------------------------------------

[1/5] REAL
    Extracting from t=2.64s to t=4.64s


/tmp/ipykernel_28080/2291482261.py:54: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(video_path, sr=CFG['sr'], offset=start_sec, duration=CFG['duration'])
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


    ✓ Video: torch.Size([50, 3, 224, 224])
    ✓ Audio: torch.Size([1, 128, 63])
    ✓ Labels: Audio=1, Video=1

[2/5] BOTH_MODIFIED
    Extracting from t=2.70s to t=4.70s


/tmp/ipykernel_28080/2291482261.py:54: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(video_path, sr=CFG['sr'], offset=start_sec, duration=CFG['duration'])
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


    ✓ Video: torch.Size([50, 3, 224, 224])
    ✓ Audio: torch.Size([1, 128, 63])
    ✓ Labels: Audio=0, Video=0

[3/5] AUDIO_MODIFIED
    Extracting from t=0.00s to t=2.00s


...

    ✓ Video: torch.Size([50, 3, 224, 224])
    ✓ Audio: torch.Size([1, 128, 63])
    ✓ Labels: Audio=0, Video=1

[4/5] VISUAL_MODIFIED
    Extracting from t=1.24s to t=3.24s


/tmp/ipykernel_28080/2291482261.py:54: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(video_path, sr=CFG['sr'], offset=start_sec, duration=CFG['duration'])
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


    ✓ Video: torch.Size([50, 3, 224, 224])
    ✓ Audio: torch.Size([1, 128, 63])
    ✓ Labels: Audio=1, Video=0

[5/5] VISUAL_MODIFIED
    Extracting from t=4.90s to t=6.90s


/tmp/ipykernel_28080/2291482261.py:54: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(video_path, sr=CFG['sr'], offset=start_sec, duration=CFG['duration'])
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


    ✓ Video: torch.Size([50, 3, 224, 224])
    ✓ Audio: torch.Size([1, 128, 63])
    ✓ Labels: Audio=1, Video=0

Feature Extraction Summary

Successfully extracted: 5/5 videos

1. REAL
   Video tensor: torch.Size([50, 3, 224, 224])
   Audio tensor: torch.Size([1, 128, 63])
   Labels: [Audio=1, Video=1]

2. BOTH_MODIFIED
   Video tensor: torch.Size([50, 3, 224, 224])
   Audio tensor: torch.Size([1, 128, 63])
   Labels: [Audio=0, Video=0]

3. AUDIO_MODIFIED
   Video tensor: torch.Size([50, 3, 224, 224])
   Audio tensor: torch.Size([1, 128, 63])
   Labels: [Audio=0, Video=1]

4. VISUAL_MODIFIED
   Video tensor: torch.Size([50, 3, 224, 224])
   Audio tensor: torch.Size([1, 128, 63])
   Labels: [Audio=1, Video=0]

5. VISUAL_MODIFIED
   Video tensor: torch.Size([50, 3, 224, 224])
   Audio tensor: torch.Size([1, 128, 63])
   Labels: [Audio=1, Video=0]

✅ Section 3 Complete - Features extracted


In [7]:
# =============================================================================
# SECTION 4: Pre-trained Encoders with Original Features
# =============================================================================

print("=" * 60)
print("SECTION 4: Pre-trained Encoders")
print("=" * 60)

class VideoEncoder(nn.Module):
    """Pre-trained ResNet3D-18 for video (50, 3, 224, 224)"""
    def __init__(self, feature_dim=256):
        super().__init__()
        self.backbone = models.video.r3d_18(pretrained=True)

        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, feature_dim)
        )

    def forward(self, x):
        # x: (B, T, C, H, W) = (B, 50, 3, 224, 224)
        # ResNet3D expects (B, C, T, H, W)
        x = x.permute(0, 2, 1, 3, 4)
        return self.backbone(x)

class AudioEncoder(nn.Module):
    """Pre-trained ResNet18 for audio spectrogram (1, 128, 87)"""
    def __init__(self, feature_dim=256):
        super().__init__()
        self.backbone = models.resnet18(pretrained=True)

        # Change first conv for 1 channel
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, feature_dim)
        )

    def forward(self, x):
        # x: (B, 1, 128, 87) -> resize to 224x224 for ResNet
        x = F.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)
        return self.backbone(x)

class CrossModalFusion(nn.Module):
    def __init__(self, feature_dim=256, hidden_dim=512, dropout=0.4):
        super().__init__()

        self.fusion = nn.Sequential(
            nn.Linear(feature_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout/2)
        )

        self.audio_classifier = nn.Linear(hidden_dim, 1)
        self.video_classifier = nn.Linear(hidden_dim, 1)
        self.joint_classifier = nn.Linear(hidden_dim, 1)

    def forward(self, video_feat, audio_feat):
        combined = torch.cat([video_feat, audio_feat], dim=1)
        fused = self.fusion(combined)

        audio_pred = torch.sigmoid(self.audio_classifier(fused))
        video_pred = torch.sigmoid(self.video_classifier(fused))
        joint_pred = torch.sigmoid(self.joint_classifier(fused))

        return {
            'audio_pred': audio_pred,
            'video_pred': video_pred,
            'joint_pred': joint_pred,
            'fused': fused
        }

class PretrainedAVDeepfakeDetector(nn.Module):
    def __init__(self, feature_dim=256, hidden_dim=512, dropout=0.4):
        super().__init__()
        self.video_encoder = VideoEncoder(feature_dim)
        self.audio_encoder = AudioEncoder(feature_dim)
        self.fusion_module = CrossModalFusion(feature_dim, hidden_dim, dropout)

    def forward(self, video, audio):
        video_feat = self.video_encoder(video)
        audio_feat = self.audio_encoder(audio)
        return self.fusion_module(video_feat, audio_feat)

model = PretrainedAVDeepfakeDetector(
    feature_dim=CONFIG['feature_dim'],
    hidden_dim=CONFIG['hidden_dim'],
    dropout=CONFIG['dropout']
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model initialized: {total_params:,} parameters")

with torch.no_grad():
    test_video = torch.randn(2, 50, 3, 224, 224).to(device)
    test_audio = torch.randn(2, 1, 128, 87).to(device)
    test_out = model(test_video, test_audio)
    print(f"✓ Test forward pass: A={test_out['audio_pred'].shape}, V={test_out['video_pred'].shape}, J={test_out['joint_pred'].shape}")

print("\n" + "=" * 60)
print("✅ Section 4 Complete - Model built")
print("=" * 60)

SECTION 4: Pre-trained Encoders


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=R3D_18_Weights.KINETICS400_V1`. You can also use `weights=R3D_18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights

Model initialized: 45,651,331 parameters
✓ Test forward pass: A=torch.Size([2, 1]), V=torch.Size([2, 1]), J=torch.Size([2, 1])

✅ Section 4 Complete - Model built


In [ ]:
# =============================================================================
# SECTION 5: W&B Setup & Progressive Training
# =============================================================================

print("=" * 60)
print("SECTION 5: Training with Progressive Fine-tuning")
print("=" * 60)

wandb.init(
    project=CONFIG['project_name'],
    name=CONFIG['run_name'],
    config=CONFIG
)

wandb.watch(model, log="all", log_freq=10)

class AVDataset(torch.utils.data.Dataset):
    def __init__(self, features):
        self.features = features
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        item = self.features[idx]
        return {
            'video': item['video'],
            'audio': item['audio'],
            'labels': item['labels'],
            'type': item['type'],
            'file': item['file']
        }

dataset = AVDataset(features)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=True)

class LabelSmoothingBCE(nn.Module):
    def __init__(self, smoothing=0.05):
        super().__init__()
        self.smoothing = smoothing
    def forward(self, pred, target):
        target = target * (1 - self.smoothing) + 0.5 * self.smoothing
        return F.binary_cross_entropy(pred, target)

criterion = LabelSmoothingBCE(smoothing=CONFIG['label_smoothing'])
criterion_hard = nn.BCELoss()

def get_optimizer(model, freeze_encoders=True):
    if freeze_encoders:
        for param in model.video_encoder.parameters():
            param.requires_grad = False
        for param in model.audio_encoder.parameters():
            param.requires_grad = False

        optimizer = optim.AdamW(
            model.fusion_module.parameters(),
            lr=CONFIG['learning_rate'],
            weight_decay=CONFIG['weight_decay']
        )
        print(f"✓ Phase 1: Frozen encoders, training fusion only ({sum(p.numel() for p in model.fusion_module.parameters() if p.requires_grad):,} params)")
    else:
        for param in model.video_encoder.parameters():
            param.requires_grad = True
        for param in model.audio_encoder.parameters():
            param.requires_grad = True

        optimizer = optim.AdamW([
            {'params': model.video_encoder.parameters(), 'lr': CONFIG['encoder_lr']},
            {'params': model.audio_encoder.parameters(), 'lr': CONFIG['encoder_lr']},
            {'params': model.fusion_module.parameters(), 'lr': CONFIG['learning_rate']}
        ], weight_decay=CONFIG['weight_decay'])
        print(f"✓ Phase 2: Fine-tuning all ({sum(p.numel() for p in model.parameters() if p.requires_grad):,} params)")

    return optimizer

optimizer = get_optimizer(model, freeze_encoders=True)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

best_loss = float('inf')
patience_counter = 0
history = {k: [] for k in ['train_loss', 'val_loss', 'train_auc', 'val_auc', 'audio_auc', 'video_auc', 'joint_auc', 'lr']}

print(f"\nStarting training: {CONFIG['epochs']} epochs (freeze for first {CONFIG['freeze_epochs']})")

for epoch in range(CONFIG['epochs']):
    if epoch == CONFIG['freeze_epochs']:
        optimizer = get_optimizer(model, freeze_encoders=False)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    model.train()
    epoch_loss = 0
    train_preds = {'audio': [], 'video': [], 'joint': []}
    train_labels = {'audio': [], 'video': [], 'joint': []}

    num_repeats = 8

    for _ in range(num_repeats):
        for batch in dataloader:
            video = batch['video'].to(device)
            audio = batch['audio'].to(device)
            labels = batch['labels'].to(device)

            audio_labels = labels[:, 0:1]
            video_labels = labels[:, 1:2]
            joint_labels = ((audio_labels == 1) & (video_labels == 1)).float()

            optimizer.zero_grad()
            outputs = model(video, audio)

            loss = (criterion(outputs['audio_pred'], audio_labels) +
                   criterion(outputs['video_pred'], video_labels) +
                   2.0 * criterion_hard(outputs['joint_pred'], joint_labels))

            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
            optimizer.step()

            epoch_loss += loss.item()
            train_preds['audio'].extend(outputs['audio_pred'].detach().cpu().numpy())
            train_preds['video'].extend(outputs['video_pred'].detach().cpu().numpy())
            train_preds['joint'].extend(outputs['joint_pred'].detach().cpu().numpy())
            train_labels['audio'].extend(audio_labels.cpu().numpy())
            train_labels['video'].extend(video_labels.cpu().numpy())
            train_labels['joint'].extend(joint_labels.cpu().numpy())

    def calc_auc(y_true, y_pred):
        y_true, y_pred = np.array(y_true).flatten(), np.array(y_pred).flatten()
        return roc_auc_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.5

    avg_train_loss = epoch_loss / (len(dataloader) * num_repeats)
    train_auc_joint = calc_auc(train_labels['joint'], train_preds['joint'])

    model.eval()
    val_loss = 0
    val_preds = {'audio': [], 'video': [], 'joint': []}
    val_labels = {'audio': [], 'video': [], 'joint': []}

    with torch.no_grad():
        for batch in dataloader:
            video = batch['video'].to(device)
            audio = batch['audio'].to(device)
            labels = batch['labels'].to(device)

            audio_labels = labels[:, 0:1]
            video_labels = labels[:, 1:2]
            joint_labels = ((audio_labels == 1) & (video_labels == 1)).float()

            outputs = model(video, audio)
            loss = (criterion_hard(outputs['audio_pred'], audio_labels) +
                   criterion_hard(outputs['video_pred'], video_labels) +
                   2.0 * criterion_hard(outputs['joint_pred'], joint_labels))

            val_loss += loss.item()
            val_preds['audio'].extend(outputs['audio_pred'].cpu().numpy())
            val_preds['video'].extend(outputs['video_pred'].cpu().numpy())
            val_preds['joint'].extend(outputs['joint_pred'].cpu().numpy())
            val_labels['audio'].extend(audio_labels.cpu().numpy())
            val_labels['video'].extend(video_labels.cpu().numpy())
            val_labels['joint'].extend(joint_labels.cpu().numpy())

    avg_val_loss = val_loss / len(dataloader)
    val_auc_audio = calc_auc(val_labels['audio'], val_preds['audio'])
    val_auc_video = calc_auc(val_labels['video'], val_preds['video'])
    val_auc_joint = calc_auc(val_labels['joint'], val_preds['joint'])
    current_lr = optimizer.param_groups[0]['lr'] if isinstance(optimizer.param_groups[0]['lr'], float) else optimizer.param_groups[0]['lr']

    for k, v in zip(['train_loss', 'val_loss', 'train_auc', 'val_auc', 'audio_auc', 'video_auc', 'joint_auc', 'lr'],
                    [avg_train_loss, avg_val_loss, train_auc_joint, val_auc_joint, val_auc_audio, val_auc_video, val_auc_joint, current_lr]):
        history[k].append(v)

    log_dict = {
        "epoch": epoch,
        "phase": "frozen" if epoch < CONFIG['freeze_epochs'] else "finetune",
        "train/loss": avg_train_loss,
        "train/auc": train_auc_joint,
        "val/loss": avg_val_loss,
        "val/auc_audio": val_auc_audio,
        "val/auc_video": val_auc_video,
        "val/auc_joint": val_auc_joint,
        "learning_rate": current_lr
    }

    if epoch % 2 == 0:
        log_dict.update({
            "predictions/audio_mean": np.mean(val_preds['audio']),
            "predictions/audio_std": np.std(val_preds['audio']),
            "predictions/video_mean": np.mean(val_preds['video']),
            "predictions/video_std": np.std(val_preds['video']),
            "predictions/joint_mean": np.mean(val_preds['joint']),
            "predictions/joint_std": np.std(val_preds['joint'])
        })

    wandb.log(log_dict)

    phase_str = "[FROZEN]" if epoch < CONFIG['freeze_epochs'] else "[FINETUNE]"
    if (epoch + 1) % 2 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:2d}/{CONFIG['epochs']}] {phase_str} "
              f"Train: {avg_train_loss:.4f}/{train_auc_joint:.3f} | "
              f"Val: {avg_val_loss:.4f}/A{val_auc_audio:.2f}/V{val_auc_video:.2f}/J{val_auc_joint:.2f}")

    scheduler.step(avg_val_loss)

    if avg_val_loss < best_loss - 0.0001:
        best_loss = avg_val_loss
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': best_loss,
            'auc_joint': val_auc_joint,
            'config': CONFIG
        }, '/tmp/best_pretrained_model.pth')
        wandb.save('/tmp/best_pretrained_model.pth')
    else:
        patience_counter += 1
        if patience_counter >= CONFIG['patience']:
            print(f"\n⏹️ Early stopping at epoch {epoch+1}")
            break

print("\n✅ Training complete")

SECTION 5: Training with Progressive Fine-tuning


epoch,▁
learning_rate,▁
predictions/audio_mean,▁
predictions/audio_std,▁
predictions/joint_mean,▁
predictions/joint_std,▁
predictions/video_mean,▁
predictions/video_std,▁
train/auc,▁
train/loss,▁
+4,...


✓ Phase 1: Frozen encoders, training fusion only (526,851 params)

Starting training: 20 epochs (freeze for first 5)
Epoch [ 1/20] [FROZEN] Train: 2.4362/0.633 | Val: 2.2159/A0.67/V1.00/J1.00


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch [ 2/20] [FROZEN] Train: 2.3557/0.598 | Val: 2.1716/A0.50/V1.00/J1.00
Epoch [ 4/20] [FROZEN] Train: 2.3155/0.473 | Val: 2.1394/A0.50/V0.83/J1.00
✓ Phase 2: Fine-tuning all (45,651,331 params)
Epoch [ 6/20] [FINETUNE] Train: 2.3228/0.781 | Val: 2.0876/A0.67/V1.00/J1.00
Epoch [ 8/20] [FINETUNE] Train: 2.0614/0.965 | Val: 1.6304/A0.67/V1.00/J1.00
Epoch [10/20] [FINETUNE] Train: 1.6398/0.996 | Val: 1.6149/A0.67/V1.00/J1.00
Epoch [12/20] [FINETUNE] Train: 2.1843/0.930 | Val: 1.1855/A0.67/V1.00/J1.00
Epoch [14/20] [FINETUNE] Train: 1.8762/0.961 | Val: 0.8575/A1.00/V1.00/J1.00


In [ ]:
# =============================================================================
# SECTION 6: Final Evaluation
# =============================================================================

print("=" * 60)
print("SECTION 6: Final Evaluation")
print("=" * 60)

checkpoint = torch.load('/tmp/best_pretrained_model.pth', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Loaded best model from epoch {checkpoint['epoch']+1}")

final_preds = []
with torch.no_grad():
    for feat in features:
        video = feat['video'].unsqueeze(0).to(device)
        audio = feat['audio'].unsqueeze(0).to(device)
        outputs = model(video, audio)

        final_preds.append({
            'file': feat['file'],
            'type': feat['type'],
            'audio_gt': feat['labels'][0].item(),
            'video_gt': feat['labels'][1].item(),
            'audio_pred': outputs['audio_pred'].item(),
            'video_pred': outputs['video_pred'].item(),
            'joint_pred': outputs['joint_pred'].item()
        })

df_final = pd.DataFrame(final_preds)

print("\nPer-sample results:")
print(df_final.to_string())

print("\nFinal Metrics:")
for mod in ['audio', 'video', 'joint']:
    if mod == 'joint':
        y_true = ((df_final['audio_gt'] == 0) | (df_final['video_gt'] == 0)).astype(int)
    else:
        y_true = df_final[f'{mod}_gt']

    y_pred = df_final[f'{mod}_pred']
    y_pred_bin = (y_pred > 0.5).astype(int)

    print(f"\n{mod.upper()}:")
    print(f"  Accuracy: {accuracy_score(y_true, y_pred_bin):.4f}")
    print(f"  AUC: {roc_auc_score(y_true, y_pred):.4f}")
    print(f"  F1: {f1_score(y_true, y_pred_bin, zero_division=0):.4f}")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes[0,0].plot(history['train_loss'], label='Train')
axes[0,0].plot(history['val_loss'], label='Val')
axes[0,0].set_title('Loss')
axes[0,0].legend()

axes[0,1].plot(history['audio_auc'], label='Audio')
axes[0,1].plot(history['video_auc'], label='Video')
axes[0,1].plot(history['joint_auc'], label='Joint')
axes[0,1].axhline(0.5, color='r', linestyle='--')
axes[0,1].set_title('AUC')
axes[0,1].legend()

axes[1,0].plot(history['lr'])
axes[1,0].set_title('Learning Rate')
axes[1,0].set_yscale('log')

colors = {'real': 'green', 'audio_modified': 'blue', 'visual_modified': 'orange', 'both_modified': 'red'}
for t in df_final['type'].unique():
    subset = df_final[df_final['type'] == t]
    axes[1,1].scatter(subset['audio_pred'], subset['video_pred'],
                     c=colors.get(t, 'gray'), label=t, s=100)
axes[1,1].plot([0,1], [0.5,0.5], 'k--', alpha=0.3)
axes[1,1].plot([0.5,0.5], [0,1], 'k--', alpha=0.3)
axes[1,1].set_xlabel('Audio')
axes[1,1].set_ylabel('Video')
axes[1,1].legend()
axes[1,1].set_title('Predictions by Type')

plt.tight_layout()
plt.savefig('/tmp/pretrained_results.png')
wandb.log({"final_plot": wandb.Image('/tmp/pretrained_results.png')})
plt.show()

wandb.summary.update({
    "best_epoch": checkpoint['epoch'],
    "best_val_loss": checkpoint['loss'],
    "best_auc_joint": checkpoint['auc_joint']
})

print(f"\nView run at: {wandb.run.url}")
wandb.finish()

## 15 Vedios- No checkpoints

In [ ]:
"""
OPTIMIZED AVDEEPFAKE1M++ PIPELINE - 15 VIDEOS
Clean W&B logging with actionable insights only
"""

import os
import json
import pickle
import random
import numpy as np
import pandas as pd
import cv2
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import torchvision.models as models
import subprocess
import sys

try:
    import wandb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "wandb", "-q"])
    import wandb

# Set seeds
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# =============================================================================
# CONFIGURATION - 15 SAMPLES
# =============================================================================

VAL_DIR = '/content/drive/MyDrive/val/extracted_val'

CONFIG = {
    "project_name": "av-deepfake-detection",
    "run_name": "15-samples-clean-metrics",
    "architecture": "PretrainedResNet3D_ResNet18",
    "dataset": "AVDeepfake1M++",
    "samples": 15,  # CHANGED: 15 videos
    "samples_per_type": {  # Balanced sampling
        "real": 4,
        "both_modified": 4,
        "audio_modified": 4,
        "visual_modified": 3
    },
    "video_encoder": "r3d_18",
    "audio_encoder": "resnet18",
    "feature_dim": 256,
    "hidden_dim": 512,
    "dropout": 0.4,
    "learning_rate": 1e-4,
    "encoder_lr": 1e-5,
    "weight_decay": 1e-4,
    "batch_size": 3,  # CHANGED: Larger batch size for 15 samples
    "epochs": 30,
    "freeze_epochs": 8,
    "patience": 12,
    "grad_clip": 1.0,
    "label_smoothing": 0.05,
    "val_split": 0.2  # CHANGED: 20% validation (3 videos)
}

# =============================================================================
# SECTION 1: Reading JSON Metadata
# =============================================================================

print("=" * 60)
print("SECTION 1: Reading JSON Metadata")
print("=" * 60)

metadata_paths = [
    os.path.join(VAL_DIR, 'val_metadata.json'),
    os.path.join(VAL_DIR, '..', 'val_metadata.json'),
    os.path.join(os.path.dirname(VAL_DIR), 'val_metadata.json'),
]

metadata_path = None
for path in metadata_paths:
    if os.path.exists(path):
        metadata_path = path
        print(f"\n✓ Found metadata at: {path}")
        break

if metadata_path is None:
    raise FileNotFoundError("Could not find val_metadata.json")

with open(metadata_path, 'r') as f:
    metadata = json.load(f)

print(f"✓ Loaded {len(metadata):,} entries")

df = pd.DataFrame(metadata)

print(f"\nDataset Info:")
print(f"  Columns: {df.columns.tolist()}")
print(f"  Shape: {df.shape}")

print(f"\nModification types:")
print(df['modify_type'].value_counts())

print("\n" + "=" * 60)
print("✅ Section 1 Complete - Metadata loaded")
print("=" * 60)

# =============================================================================
# SECTION 2: Sampling 15 Videos (Balanced + Train/Val Split)
# =============================================================================

print("=" * 60)
print("SECTION 2: Sampling 15 Videos with Train/Val Split")
print("=" * 60)

random.seed(42)
np.random.seed(42)

df_with_audio = df[df['audio_frames'] > 0].copy()
print(f"\nVideos with audio: {len(df_with_audio):,}")

# Sample balanced dataset
samples = []
for mod_type, count in CONFIG['samples_per_type'].items():
    subset = df_with_audio[df_with_audio['modify_type'] == mod_type]
    if len(subset) >= count:
        sampled = subset.sample(count, random_state=42)
        samples.append(sampled)
        print(f"✓ {mod_type:20s}: {count} samples")
    else:
        print(f"⚠ {mod_type:20s}: only {len(subset)} available, taking all")
        samples.append(subset)

mini_df = pd.concat(samples).reset_index(drop=True)

# Create train/val split (stratified by type)
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(
    mini_df,
    test_size=CONFIG['val_split'],
    stratify=mini_df['modify_type'],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"\n{'='*60}")
print("Split Summary:")
print(f"  Train: {len(train_df)} videos")
print(f"  Val:   {len(val_df)} videos")

print(f"\nTrain distribution:")
print(train_df['modify_type'].value_counts())
print(f"\nVal distribution:")
print(val_df['modify_type'].value_counts())

print(f"\n" + "=" * 60)
print("Selected Videos Summary:")
print("=" * 60)

for split_name, split_df in [("TRAIN", train_df), ("VAL", val_df)]:
    print(f"\n{split_name} SET ({len(split_df)} videos):")
    for i, row in split_df.iterrows():
        print(f"  {i+1}. {row['modify_type'][:20]:20s} | {row['file'][:50]:50s}")

with open('/tmp/train_df.pkl', 'wb') as f:
    pickle.dump(train_df, f)
with open('/tmp/val_df.pkl', 'wb') as f:
    pickle.dump(val_df, f)

print("\n" + "=" * 60)
print("✅ Section 2 Complete - 15 videos selected with train/val split")
print("=" * 60)

# =============================================================================
# SECTION 3: Audio-Visual Feature Extraction
# =============================================================================

print("=" * 60)
print("SECTION 3: Feature Extraction")
print("=" * 60)

CFG = {
    'sr': 16000,
    'fps': 25,
    'duration': 2.0,
    'num_frames': 50,
    'img_size': 224,
    'audio_samples': 32000
}

def extract_av_features(video_path, fake_segments=None, total_frames=0):
    if fake_segments and len(fake_segments) > 0:
        start_sec = fake_segments[0][0]
    else:
        total_sec = total_frames / CFG['fps']
        start_sec = max(0, (total_sec / 2) - (CFG['duration'] / 2))

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None, None

    start_frame = int(start_sec * CFG['fps'])
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    frames = []
    for _ in range(CFG['num_frames']):
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (CFG['img_size'], CFG['img_size']))
        frame = frame / 255.0
        frames.append(frame)
    cap.release()

    if len(frames) < CFG['num_frames']:
        while len(frames) < CFG['num_frames']:
            frames.append(frames[-1] if frames else np.zeros((224, 224, 3)))

    video_tensor = torch.FloatTensor(np.array(frames)).permute(0, 3, 1, 2)

    try:
        y, sr = librosa.load(video_path, sr=CFG['sr'], offset=start_sec, duration=CFG['duration'])
        if len(y) < CFG['audio_samples']:
            y = np.pad(y, (0, CFG['audio_samples'] - len(y)))
        else:
            y = y[:CFG['audio_samples']]

        mel_spec = librosa.feature.melspectrogram(y=y, sr=CFG['sr'], n_mels=128, n_fft=2048, hop_length=512)
        mel_db = librosa.power_to_db(mel_spec, ref=np.max)
        audio_tensor = torch.FloatTensor(mel_db).unsqueeze(0)

    except Exception as e:
        print(f"    ⚠ Audio failed: {e}")
        audio_tensor = torch.zeros(1, 128, 87)

    return video_tensor, audio_tensor

def process_split(split_df, split_name):
    print(f"\nProcessing {split_name} set ({len(split_df)} videos)...")
    features = []

    for idx, row in split_df.iterrows():
        print(f"\n[{split_name} {idx+1}/{len(split_df)}] {row['modify_type'].upper()}")
        video_path = os.path.join(VAL_DIR, row['file'])

        if not os.path.exists(video_path):
            print(f"    ✗ File not found")
            continue

        video_tensor, audio_tensor = extract_av_features(
            video_path=video_path,
            fake_segments=row.get('fake_segments'),
            total_frames=row['video_frames']
        )

        if video_tensor is None:
            continue

        mt = row['modify_type']
        if mt == 'real':
            audio_label, video_label = 1, 1
        elif mt == 'both_modified':
            audio_label, video_label = 0, 0
        elif mt == 'audio_modified':
            audio_label, video_label = 0, 1
        else:
            audio_label, video_label = 1, 0

        features.append({
            'video': video_tensor,
            'audio': audio_tensor,
            'labels': torch.FloatTensor([audio_label, video_label]),
            'type': mt,
            'file': row['file']
        })

        print(f"    ✓ Video: {video_tensor.shape}, Audio: {audio_tensor.shape}")

    return features

train_features = process_split(train_df, "TRAIN")
val_features = process_split(val_df, "VAL")

print(f"\n{'='*60}")
print("Extraction Summary:")
print(f"  Train: {len(train_features)}/{len(train_df)} successful")
print(f"  Val:   {len(val_features)}/{len(val_df)} successful")

with open('/tmp/train_features.pkl', 'wb') as f:
    pickle.dump(train_features, f)
with open('/tmp/val_features.pkl', 'wb') as f:
    pickle.dump(val_features, f)

print("\n" + "=" * 60)
print("✅ Section 3 Complete - Features extracted")
print("=" * 60)

# =============================================================================
# SECTION 4: Pre-trained Encoders
# =============================================================================

print("=" * 60)
print("SECTION 4: Pre-trained Encoders")
print("=" * 60)

class VideoEncoder(nn.Module):
    def __init__(self, feature_dim=256):
        super().__init__()
        self.backbone = models.video.r3d_18(pretrained=True)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, feature_dim)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1, 3, 4)
        return self.backbone(x)

class AudioEncoder(nn.Module):
    def __init__(self, feature_dim=256):
        super().__init__()
        self.backbone = models.resnet18(pretrained=True)
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, feature_dim)
        )

    def forward(self, x):
        x = F.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)
        return self.backbone(x)

class CrossModalFusion(nn.Module):
    def __init__(self, feature_dim=256, hidden_dim=512, dropout=0.4):
        super().__init__()
        self.fusion = nn.Sequential(
            nn.Linear(feature_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout/2)
        )
        self.audio_classifier = nn.Linear(hidden_dim, 1)
        self.video_classifier = nn.Linear(hidden_dim, 1)
        self.joint_classifier = nn.Linear(hidden_dim, 1)

    def forward(self, video_feat, audio_feat):
        combined = torch.cat([video_feat, audio_feat], dim=1)
        fused = self.fusion(combined)
        return {
            'audio_pred': torch.sigmoid(self.audio_classifier(fused)),
            'video_pred': torch.sigmoid(self.video_classifier(fused)),
            'joint_pred': torch.sigmoid(self.joint_classifier(fused)),
            'fused': fused
        }

class PretrainedAVDeepfakeDetector(nn.Module):
    def __init__(self, feature_dim=256, hidden_dim=512, dropout=0.4):
        super().__init__()
        self.video_encoder = VideoEncoder(feature_dim)
        self.audio_encoder = AudioEncoder(feature_dim)
        self.fusion_module = CrossModalFusion(feature_dim, hidden_dim, dropout)

    def forward(self, video, audio):
        video_feat = self.video_encoder(video)
        audio_feat = self.audio_encoder(audio)
        return self.fusion_module(video_feat, audio_feat)

model = PretrainedAVDeepfakeDetector(
    feature_dim=CONFIG['feature_dim'],
    hidden_dim=CONFIG['hidden_dim'],
    dropout=CONFIG['dropout']
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {total_params:,} parameters")

with torch.no_grad():
    test_video = torch.randn(2, 50, 3, 224, 224).to(device)
    test_audio = torch.randn(2, 1, 128, 87).to(device)
    test_out = model(test_video, test_audio)
    print(f"✓ Test: A={test_out['audio_pred'].shape}, V={test_out['video_pred'].shape}, J={test_out['joint_pred'].shape}")

print("\n" + "=" * 60)
print("✅ Section 4 Complete - Model built")
print("=" * 60)

# =============================================================================
# SECTION 5: W&B Setup & Training (CLEAN METRICS)
# =============================================================================

print("=" * 60)
print("SECTION 5: Training with Clean W&B Logging")
print("=" * 60)

wandb.init(
    project=CONFIG['project_name'],
    name=CONFIG['run_name'],
    config=CONFIG
)

# Only log model graph, not all gradients (reduces overhead)
wandb.watch(model, log="parameters", log_freq=100)

class AVDataset(torch.utils.data.Dataset):
    def __init__(self, features):
        self.features = features
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        item = self.features[idx]
        return {
            'video': item['video'],
            'audio': item['audio'],
            'labels': item['labels'],
            'type': item['type'],
            'file': item['file']
        }

train_dataset = AVDataset(train_features)
val_dataset = AVDataset(val_features)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    drop_last=True  # Ensure consistent batch sizes
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False
)

class LabelSmoothingBCE(nn.Module):
    def __init__(self, smoothing=0.05):
        super().__init__()
        self.smoothing = smoothing
    def forward(self, pred, target):
        target = target * (1 - self.smoothing) + 0.5 * self.smoothing
        return F.binary_cross_entropy(pred, target)

criterion = LabelSmoothingBCE(smoothing=CONFIG['label_smoothing'])
criterion_hard = nn.BCELoss()

def get_optimizer(model, freeze_encoders=True):
    if freeze_encoders:
        for param in model.video_encoder.parameters():
            param.requires_grad = False
        for param in model.audio_encoder.parameters():
            param.requires_grad = False

        optimizer = optim.AdamW(
            model.fusion_module.parameters(),
            lr=CONFIG['learning_rate'],
            weight_decay=CONFIG['weight_decay']
        )
        trainable = sum(p.numel() for p in model.fusion_module.parameters() if p.requires_grad)
        print(f"✓ Phase 1: Frozen encoders, {trainable:,} params trainable")
    else:
        for param in model.video_encoder.parameters():
            param.requires_grad = True
        for param in model.audio_encoder.parameters():
            param.requires_grad = True

        optimizer = optim.AdamW([
            {'params': model.video_encoder.parameters(), 'lr': CONFIG['encoder_lr']},
            {'params': model.audio_encoder.parameters(), 'lr': CONFIG['encoder_lr']},
            {'params': model.fusion_module.parameters(), 'lr': CONFIG['learning_rate']}
        ], weight_decay=CONFIG['weight_decay'])
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"✓ Phase 2: Fine-tuning all, {trainable:,} params trainable")

    return optimizer

optimizer = get_optimizer(model, freeze_encoders=True)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

# ESSENTIAL METRICS ONLY - removed redundant tracking
best_val_auc = 0.0
patience_counter = 0

print(f"\nTraining: {CONFIG['epochs']} epochs (freeze first {CONFIG['freeze_epochs']})")
print(f"Batch size: {CONFIG['batch_size']}, Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

for epoch in range(CONFIG['epochs']):
    # Phase transition
    if epoch == CONFIG['freeze_epochs']:
        optimizer = get_optimizer(model, freeze_encoders=False)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    # TRAINING
    model.train()
    train_loss = 0.0

    for batch in train_loader:
        video = batch['video'].to(device)
        audio = batch['audio'].to(device)
        labels = batch['labels'].to(device)

        audio_labels = labels[:, 0:1]
        video_labels = labels[:, 1:2]
        joint_labels = ((audio_labels == 1) & (video_labels == 1)).float()

        optimizer.zero_grad()
        outputs = model(video, audio)

        loss = (criterion(outputs['audio_pred'], audio_labels) +
               criterion(outputs['video_pred'], video_labels) +
               2.0 * criterion_hard(outputs['joint_pred'], joint_labels))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # VALIDATION (with detailed metrics)
    model.eval()
    val_loss = 0.0
    all_preds = {'audio': [], 'video': [], 'joint': []}
    all_labels = {'audio': [], 'video': [], 'joint': []}
    all_types = []

    with torch.no_grad():
        for batch in val_loader:
            video = batch['video'].to(device)
            audio = batch['audio'].to(device)
            labels = batch['labels'].to(device)

            audio_labels = labels[:, 0:1]
            video_labels = labels[:, 1:2]
            joint_labels = ((audio_labels == 1) & (video_labels == 1)).float()

            outputs = model(video, audio)

            loss = (criterion_hard(outputs['audio_pred'], audio_labels) +
                   criterion_hard(outputs['video_pred'], video_labels) +
                   2.0 * criterion_hard(outputs['joint_pred'], joint_labels))

            val_loss += loss.item()

            all_preds['audio'].extend(outputs['audio_pred'].cpu().numpy())
            all_preds['video'].extend(outputs['video_pred'].cpu().numpy())
            all_preds['joint'].extend(outputs['joint_pred'].cpu().numpy())
            all_labels['audio'].extend(audio_labels.cpu().numpy())
            all_labels['video'].extend(video_labels.cpu().numpy())
            all_labels['joint'].extend(joint_labels.cpu().numpy())
            all_types.extend(batch['type'])

    avg_val_loss = val_loss / len(val_loader)

    # Calculate AUCs
    def calc_auc(y_true, y_pred):
        y_true = np.array(y_true).flatten()
        y_pred = np.array(y_pred).flatten()
        return roc_auc_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.5

    val_auc_audio = calc_auc(all_labels['audio'], all_preds['audio'])
    val_auc_video = calc_auc(all_labels['video'], all_preds['video'])
    val_auc_joint = calc_auc(all_labels['joint'], all_preds['joint'])

    # Per-type accuracy (key insight for deepfake detection)
    type_correct = {}
    type_total = {}
    for i, t in enumerate(all_types):
        audio_correct = ((all_preds['audio'][i] > 0.5) == (all_labels['audio'][i] == 1))
        video_correct = ((all_preds['video'][i] > 0.5) == (all_labels['video'][i] == 1))
        joint_correct = ((all_preds['joint'][i] > 0.5) == (all_labels['joint'][i] == 1))

        if t not in type_correct:
            type_correct[t] = {'audio': 0, 'video': 0, 'joint': 0}
            type_total[t] = 0
        type_correct[t]['audio'] += audio_correct
        type_correct[t]['video'] += video_correct
        type_correct[t]['joint'] += joint_correct
        type_total[t] += 1

    # Log ONLY essential metrics to W&B
    log_dict = {
        "epoch": epoch,
        "phase": "frozen" if epoch < CONFIG['freeze_epochs'] else "finetune",

        # Core metrics (loss + AUC)
        "train/loss": avg_train_loss,
        "val/loss": avg_val_loss,
        "val/auc_joint": val_auc_joint,  # Main metric

        # Modality comparison (detects which is weaker)
        "val/auc_audio": val_auc_audio,
        "val/auc_video": val_auc_video,
        "val/auc_gap": abs(val_auc_audio - val_auc_video),  # Insight: are they balanced?

        # Learning dynamics
        "train/loss_ratio": avg_train_loss / (avg_val_loss + 1e-8),  # Overfitting detector
        "learning_rate": optimizer.param_groups[0]['lr'] if isinstance(optimizer.param_groups[0]['lr'], float) else optimizer.param_groups[0]['lr'],
    }

    # Per-type accuracy (logged every 3 epochs to reduce noise)
    if epoch % 3 == 0:
        for t in type_correct:
            for mod in ['audio', 'video', 'joint']:
                acc = type_correct[t][mod] / type_total[t]
                log_dict[f"acc/{t}/{mod}"] = acc

    wandb.log(log_dict)

    # Console output (concise)
    phase_str = "[FROZEN]" if epoch < CONFIG['freeze_epochs'] else "[FINETUNE]"
    if (epoch + 1) % 3 == 0 or epoch == 0:
        print(f"Ep {epoch+1:2d}/{CONFIG['epochs']} {phase_str} | "
              f"Loss: T{avg_train_loss:.3f}/V{avg_val_loss:.3f} | "
              f"AUC: J{val_auc_joint:.2f} A{val_auc_audio:.2f} V{val_auc_video:.2f} | "
              f"Gap: {abs(val_auc_audio - val_auc_video):.2f}")

    # Scheduler step
    scheduler.step(avg_val_loss)

    # Checkpoint on AUC (not loss) - more relevant for imbalanced data
    if val_auc_joint > best_val_auc:
        best_val_auc = val_auc_joint
        patience_counter = 0

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'auc_joint': val_auc_joint,
            'auc_audio': val_auc_audio,
            'auc_video': val_auc_video,
            'config': CONFIG
        }, '/tmp/best_model_15.pth')
        wandb.save('/tmp/best_model_15.pth')
    else:
        patience_counter += 1
        if patience_counter >= CONFIG['patience']:
            print(f"\n⏹️ Early stopping at epoch {epoch+1} (best AUC: {best_val_auc:.3f})")
            break

print("\n✅ Training complete")

# =============================================================================
# SECTION 6: Final Evaluation (SINGLE INSIGHTFUL VISUALIZATION)
# =============================================================================

print("=" * 60)
print("SECTION 6: Final Evaluation")
print("=" * 60)

checkpoint = torch.load('/tmp/best_model_15.pth', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Best model: epoch {checkpoint['epoch']+1}, AUC={checkpoint['auc_joint']:.3f}")

# Collect predictions
all_results = []
for split_name, split_loader in [("train", train_loader), ("val", val_loader)]:
    with torch.no_grad():
        for batch in split_loader:
            video = batch['video'].to(device)
            audio = batch['audio'].to(device)
            labels = batch['labels']

            outputs = model(video, audio)

            for i in range(len(batch['file'])):
                all_results.append({
                    'split': split_name,
                    'file': batch['file'][i],
                    'type': batch['type'][i],
                    'audio_gt': labels[i, 0].item(),
                    'video_gt': labels[i, 1].item(),
                    'audio_pred': outputs['audio_pred'][i].item(),
                    'video_pred': outputs['video_pred'][i].item(),
                    'joint_pred': outputs['joint_pred'][i].item()
                })

df_results = pd.DataFrame(all_results)

# Create SINGLE comprehensive figure (not 4 separate plots)
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Scatter: Audio vs Video predictions (main insight)
ax1 = axes[0, 0]
colors = {'real': '#2ecc71', 'audio_modified': '#3498db',
          'visual_modified': '#e67e22', 'both_modified': '#e74c3c'}
markers = {'train': 'o', 'val': 's'}

for split in ['train', 'val']:
    for t in df_results['type'].unique():
        subset = df_results[(df_results['type'] == t) & (df_results['split'] == split)]
        ax1.scatter(subset['audio_pred'], subset['video_pred'],
                   c=colors.get(t, 'gray'),
                   marker=markers[split],
                   s=150 if split == 'val' else 80,
                   alpha=0.8 if split == 'val' else 0.5,
                   edgecolors='black' if split == 'val' else 'none',
                   label=f"{t} ({split})")

ax1.axhline(y=0.5, color='k', linestyle='--', alpha=0.3)
ax1.axvline(x=0.5, color='k', linestyle='--', alpha=0.3)
ax1.set_xlabel('Audio Prediction (0=Fake, 1=Real)', fontsize=11)
ax1.set_ylabel('Video Prediction (0=Fake, 1=Real)', fontsize=11)
ax1.set_title('Audio-Video Prediction Space\n(Val=squares with border, Train=circles)', fontsize=12, fontweight='bold')
ax1.set_xlim(-0.05, 1.05)
ax1.set_ylim(-0.05, 1.05)
ax1.grid(True, alpha=0.3)

# Custom legend (avoid duplicate labels)
handles, labels = ax1.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax1.legend(by_label.values(), by_label.keys(), loc='upper left', fontsize=8)

# 2. Per-type accuracy comparison
ax2 = axes[0, 1]
type_stats = []
for t in df_results['type'].unique():
    subset = df_results[df_results['type'] == t]
    for mod in ['audio', 'video', 'joint']:
        gt_col = f'{mod}_gt' if mod != 'joint' else None
        if mod == 'joint':
            y_true = ((subset['audio_gt'] == 0) | (subset['video_gt'] == 0)).astype(int)
        else:
            y_true = subset[f'{mod}_gt']
        y_pred = (subset[f'{mod}_pred'] > 0.5).astype(int)
        acc = accuracy_score(y_true, y_pred)
        type_stats.append({'type': t, 'modality': mod, 'accuracy': acc})

df_stats = pd.DataFrame(type_stats)
pivot_stats = df_stats.pivot(index='type', columns='modality', values='accuracy')

pivot_stats.plot(kind='bar', ax=ax2, color=['#3498db', '#e67e22', '#2ecc71'])
ax2.set_title('Per-Type Accuracy by Modality', fontsize=12, fontweight='bold')
ax2.set_ylabel('Accuracy')
ax2.set_ylim(0, 1.1)
ax2.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Random')
ax2.legend(loc='lower right')
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 3. Prediction confidence distribution (detects calibration issues)
ax3 = axes[1, 0]
val_only = df_results[df_results['split'] == 'val']
ax3.hist(val_only['joint_pred'], bins=15, alpha=0.7, color='purple', edgecolor='black')
ax3.axvline(x=0.5, color='r', linestyle='--', label='Decision boundary')
ax3.set_xlabel('Joint Prediction Score')
ax3.set_ylabel('Count')
ax3.set_title('Validation Prediction Distribution\n(Good: peaks near 0 and 1, Bad: peak at 0.5)',
              fontsize=12, fontweight='bold')
ax3.legend()

# 4. Confusion matrix for joint predictions
ax4 = axes[1, 1]
val_only = df_results[df_results['split'] == 'val']
y_true = ((val_only['audio_gt'] == 0) | (val_only['video_gt'] == 0)).astype(int)
y_pred = (val_only['joint_pred'] > 0.5).astype(int)
cm = confusion_matrix(y_true, y_pred)

im = ax4.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax4.set_title('Confusion Matrix (Validation)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax4, fraction=0.046, pad=0.04)

tick_marks = np.arange(2)
ax4.set_xticks(tick_marks)
ax4.set_yticks(tick_marks)
ax4.set_xticklabels(['Predicted Fake', 'Predicted Real'])
ax4.set_yticklabels(['Actual Fake', 'Actual Real'])

# Add text annotations
thresh = cm.max() / 2.
for i in range(2):
    for j in range(2):
        ax4.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/final_analysis_15.png', dpi=150, bbox_inches='tight', facecolor='white')
wandb.log({"final_analysis": wandb.Image('/tmp/final_analysis_15.png')})
plt.show()

# Print summary table
print("\n" + "=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)

print(f"\nBest Validation AUC: {checkpoint['auc_joint']:.3f}")
print(f"Audio AUC: {checkpoint['auc_audio']:.3f}")
print(f"Video AUC: {checkpoint['auc_video']:.3f}")

print(f"\nPer-Type Accuracy (Validation):")
val_stats = df_results[df_results['split'] == 'val'].groupby('type').apply(
    lambda x: pd.Series({
        'audio_acc': accuracy_score(x['audio_gt'], (x['audio_pred'] > 0.5).astype(int)),
        'video_acc': accuracy_score(x['video_gt'], (x['video_pred'] > 0.5).astype(int)),
        'joint_acc': accuracy_score(
            ((x['audio_gt'] == 0) | (x['video_gt'] == 0)).astype(int),
            (x['joint_pred'] > 0.5).astype(int)
        ),
        'count': len(x)
    })
)
print(val_stats.to_string())

# Log final summary
wandb.summary.update({
    "best_epoch": checkpoint['epoch'],
    "best_auc_joint": checkpoint['auc_joint'],
    "best_auc_audio": checkpoint['auc_audio'],
    "best_auc_video": checkpoint['auc_video'],
    "final_val_samples": len(val_features),
    "final_train_samples": len(train_features)
})

print(f"\nView run at: {wandb.run.url}")
wandb.finish()

## 15 Videos- Checkpoints added

In [ ]:
"""
AVDEEPFAKE1M++ PIPELINE WITH CHECKPOINT RESUMPTION
Saves progress every epoch, auto-resumes if interrupted
"""

import os
import json
import pickle
import random
import numpy as np
import pandas as pd
import cv2
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import torchvision.models as models
import subprocess
import sys

try:
    import wandb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "wandb", "-q"])
    import wandb

# Set seeds
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# =============================================================================
# CONFIGURATION & CHECKPOINT PATHS
# =============================================================================

VAL_DIR = '/content/drive/MyDrive/val/extracted_val'

# Checkpoint paths - modify these to use Google Drive for persistence
CHECKPOINT_DIR = '/content/drive/MyDrive/checkpoints'  # Change to your path
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

CONFIG = {
    "project_name": "av-deepfake-detection",
    "run_name": "15-samples-resumable",
    "architecture": "PretrainedResNet3D_ResNet18",
    "dataset": "AVDeepfake1M++",
    "samples": 15,
    "samples_per_type": {
        "real": 4,
        "both_modified": 4,
        "audio_modified": 4,
        "visual_modified": 3
    },
    "feature_dim": 256,
    "hidden_dim": 512,
    "dropout": 0.4,
    "learning_rate": 1e-4,
    "encoder_lr": 1e-5,
    "weight_decay": 1e-4,
    "batch_size": 3,
    "epochs": 30,
    "freeze_epochs": 8,
    "patience": 12,
    "grad_clip": 1.0,
    "label_smoothing": 0.05,
    "val_split": 0.2,
    "checkpoint_freq": 1,  # Save every N epochs
    "resume": True  # Auto-resume if checkpoint exists
}

# File paths
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'training_checkpoint.pth')
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, 'best_model.pth')
FEATURES_TRAIN_PATH = os.path.join(CHECKPOINT_DIR, 'train_features.pkl')
FEATURES_VAL_PATH = os.path.join(CHECKPOINT_DIR, 'val_features.pkl')
WAND_ID_PATH = os.path.join(CHECKPOINT_DIR, 'wandb_run_id.txt')

# =============================================================================
# CHECKPOINT UTILITIES
# =============================================================================

def save_checkpoint(epoch, model, optimizer, scheduler, history, best_val_auc,
                   patience_counter, wandb_run_id, is_best=False):
    """Save complete training state"""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'history': history,
        'best_val_auc': best_val_auc,
        'patience_counter': patience_counter,
        'config': CONFIG,
        'wandb_run_id': wandb_run_id,
        'random_state': {
            'python': random.getstate(),
            'numpy': np.random.get_state(),
            'torch': torch.get_rng_state(),
            'torch_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        }
    }

    # Save latest checkpoint
    torch.save(checkpoint, CHECKPOINT_PATH)
    print(f"  ✓ Checkpoint saved: epoch {epoch+1}")

    # Save best model separately
    if is_best:
        torch.save(checkpoint, BEST_MODEL_PATH)
        print(f"  ✓ Best model saved (AUC: {best_val_auc:.3f})")

    # Save W&B run ID
    if wandb_run_id:
        with open(WAND_ID_PATH, 'w') as f:
            f.write(wandb_run_id)

    return checkpoint

def load_checkpoint(model):
    """Load checkpoint if it exists"""
    if not os.path.exists(CHECKPOINT_PATH):
        print("No checkpoint found. Starting from scratch.")
        return None

    print(f"Loading checkpoint from {CHECKPOINT_PATH}")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)

    # Load model weights
    model.load_state_dict(checkpoint['model_state_dict'])

    # Restore random states for reproducibility
    random.setstate(checkpoint['random_state']['python'])
    np.random.set_state(checkpoint['random_state']['numpy'])
    torch.set_rng_state(checkpoint['random_state']['torch'])
    if checkpoint['random_state']['torch_cuda'] and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(checkpoint['random_state']['torch_cuda'])

    print(f"  ✓ Resumed from epoch {checkpoint['epoch']+1}")
    print(f"  ✓ Best AUC so far: {checkpoint['best_val_auc']:.3f}")

    return checkpoint  # Return full checkpoint, not just model

def check_existing_features():
    """Check if features were already extracted"""
    if os.path.exists(FEATURES_TRAIN_PATH) and os.path.exists(FEATURES_VAL_PATH):
        print("Found pre-extracted features. Loading...")
        with open(FEATURES_TRAIN_PATH, 'rb') as f:
            train_features = pickle.load(f)
        with open(FEATURES_VAL_PATH, 'rb') as f:
            val_features = pickle.load(f)
        print(f"  ✓ Loaded {len(train_features)} train, {len(val_features)} val features")
        return train_features, val_features
    return None, None

# =============================================================================
# SECTION 1-3: DATA PROCESSING (with caching)
# =============================================================================

print("=" * 60)
print("SECTION 1-3: Data Loading & Feature Extraction")
print("=" * 60)

# Try to load cached features first
train_features, val_features = check_existing_features()

if train_features is None:
    print("\nExtracting features from scratch...")

    # Load metadata
    metadata_path = os.path.join(VAL_DIR, 'val_metadata.json')
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)

    df = pd.DataFrame(metadata)
    df_with_audio = df[df['audio_frames'] > 0].copy()

    # Sample balanced dataset
    samples = []
    for mod_type, count in CONFIG['samples_per_type'].items():
        subset = df_with_audio[df_with_audio['modify_type'] == mod_type]
        if len(subset) >= count:
            sampled = subset.sample(count, random_state=42)
            samples.append(sampled)

    mini_df = pd.concat(samples).reset_index(drop=True)

    # Train/val split
    train_df, val_df = train_test_split(
        mini_df, test_size=CONFIG['val_split'],
        stratify=mini_df['modify_type'], random_state=42
    )

    # Feature extraction config
    CFG = {'sr': 16000, 'fps': 25, 'duration': 2.0, 'num_frames': 50,
           'img_size': 224, 'audio_samples': 32000}

    def extract_features(split_df, split_name):
        features = []
        for idx, row in split_df.iterrows():
            video_path = os.path.join(VAL_DIR, row['file'])
            if not os.path.exists(video_path):
                continue

            # Extract video
            cap = cv2.VideoCapture(video_path)
            frames = []
            for _ in range(CFG['num_frames']):
                ret, frame = cap.read()
                if not ret: break
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, (CFG['img_size'], CFG['img_size']))
                frames.append(frame / 255.0)
            cap.release()

            while len(frames) < CFG['num_frames']:
                frames.append(frames[-1] if frames else np.zeros((224, 224, 3)))

            video_tensor = torch.FloatTensor(np.array(frames)).permute(0, 3, 1, 2)

            # Extract audio
            try:
                y, sr = librosa.load(video_path, sr=CFG['sr'], offset=0, duration=CFG['duration'])
                if len(y) < CFG['audio_samples']:
                    y = np.pad(y, (0, CFG['audio_samples'] - len(y)))
                mel = librosa.feature.melspectrogram(y=y, sr=CFG['sr'], n_mels=128)
                mel_db = librosa.power_to_db(mel, ref=np.max)
                audio_tensor = torch.FloatTensor(mel_db).unsqueeze(0)
            except:
                audio_tensor = torch.zeros(1, 128, 87)

            # Labels
            mt = row['modify_type']
            labels = {'real': (1,1), 'both_modified': (0,0),
                     'audio_modified': (0,1), 'visual_modified': (1,0)}[mt]

            features.append({
                'video': video_tensor, 'audio': audio_tensor,
                'labels': torch.FloatTensor(labels),
                'type': mt, 'file': row['file']
            })
            print(f"  {split_name} {idx+1}/{len(split_df)}: {mt}")

        return features

    train_features = extract_features(train_df, "TRAIN")
    val_features = extract_features(val_df, "VAL")

    # Cache features
    with open(FEATURES_TRAIN_PATH, 'wb') as f:
        pickle.dump(train_features, f)
    with open(FEATURES_VAL_PATH, 'wb') as f:
        pickle.dump(val_features, f)
    print(f"\n✓ Features cached to {CHECKPOINT_DIR}")

print(f"\nFinal: {len(train_features)} train, {len(val_features)} val samples")

# =============================================================================
# SECTION 4: MODEL (always rebuild, weights loaded from checkpoint if exists)
# =============================================================================

print("=" * 60)
print("SECTION 4: Model Architecture")
print("=" * 60)

class VideoEncoder(nn.Module):
    def __init__(self, feature_dim=256):
        super().__init__()
        self.backbone = models.video.r3d_18(pretrained=True)
        in_feat = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_feat, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, feature_dim)
        )
    def forward(self, x):
        return self.backbone(x.permute(0, 2, 1, 3, 4))

class AudioEncoder(nn.Module):
    def __init__(self, feature_dim=256):
        super().__init__()
        self.backbone = models.resnet18(pretrained=True)
        self.backbone.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
        in_feat = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_feat, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, feature_dim)
        )
    def forward(self, x):
        x = F.interpolate(x, (224, 224), mode='bilinear', align_corners=False)
        return self.backbone(x)

class AVDetector(nn.Module):
    def __init__(self, feature_dim=256, hidden_dim=512, dropout=0.4):
        super().__init__()
        self.video_enc = VideoEncoder(feature_dim)
        self.audio_enc = AudioEncoder(feature_dim)
        self.fusion = nn.Sequential(
            nn.Linear(feature_dim*2, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout/2)
        )
        self.audio_clf = nn.Linear(hidden_dim, 1)
        self.video_clf = nn.Linear(hidden_dim, 1)
        self.joint_clf = nn.Linear(hidden_dim, 1)

    def forward(self, video, audio):
        v_feat = self.video_enc(video)
        a_feat = self.audio_enc(audio)
        fused = self.fusion(torch.cat([v_feat, a_feat], 1))
        return {
            'audio_pred': torch.sigmoid(self.audio_clf(fused)),
            'video_pred': torch.sigmoid(self.video_clf(fused)),
            'joint_pred': torch.sigmoid(self.joint_clf(fused))
        }

model = AVDetector(CONFIG['feature_dim'], CONFIG['hidden_dim'], CONFIG['dropout']).to(device)
print(f"Model: {sum(p.numel() for p in model.parameters()):,} params")

In [ ]:
# =============================================================================
# SECTION 5: RESUMABLE TRAINING
# =============================================================================

print("=" * 60)
print("SECTION 5: Training (Resumable)")
print("=" * 60)

# Datasets
class AVDataset(torch.utils.data.Dataset):
    def __init__(self, features): self.features = features
    def __len__(self): return len(self.features)
    def __getitem__(self, idx):
        f = self.features[idx]
        return {'video': f['video'], 'audio': f['audio'],
                'labels': f['labels'], 'type': f['type'], 'file': f['file']}

train_loader = torch.utils.data.DataLoader(
    AVDataset(train_features), batch_size=CONFIG['batch_size'],
    shuffle=True, drop_last=True)
val_loader = torch.utils.data.DataLoader(
    AVDataset(val_features), batch_size=CONFIG['batch_size'], shuffle=False)

# Loss functions
class SmoothBCE(nn.Module):
    def __init__(self, s=0.05):
        super().__init__()
        self.s = s
    def forward(self, p, t):
        return F.binary_cross_entropy(p, t*(1-self.s) + 0.5*self.s)

criterion = SmoothBCE(CONFIG['label_smoothing'])
criterion_hard = nn.BCELoss()

# Optimizer setup function
def make_optimizer(model, freeze=True):
    if freeze:
        for p in model.video_enc.parameters(): p.requires_grad = False
        for p in model.audio_enc.parameters(): p.requires_grad = False
        opt = optim.AdamW(model.fusion.parameters(), lr=CONFIG['learning_rate'],
                         weight_decay=CONFIG['weight_decay'])
        print(f"  Frozen: {sum(p.numel() for p in model.fusion.parameters() if p.requires_grad):,} params")
    else:
        for p in model.parameters(): p.requires_grad = True
        opt = optim.AdamW([
            {'params': model.video_enc.parameters(), 'lr': CONFIG['encoder_lr']},
            {'params': model.audio_enc.parameters(), 'lr': CONFIG['encoder_lr']},
            {'params': model.fusion.parameters(), 'lr': CONFIG['learning_rate']}
        ], weight_decay=CONFIG['weight_decay'])
        print(f"  Unfrozen: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} params")
    return opt

# Try to load checkpoint
checkpoint = load_checkpoint(model) if CONFIG['resume'] else None

# Initialize or restore training state
if checkpoint:
    start_epoch = checkpoint['epoch'] + 1
    history = checkpoint['history']
    best_val_auc = checkpoint['best_val_auc']
    patience_counter = checkpoint['patience_counter']

    # Restore optimizer and scheduler
    current_phase_frozen = (start_epoch < CONFIG['freeze_epochs'])
    optimizer = make_optimizer(model, freeze=current_phase_frozen)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    if checkpoint['scheduler_state_dict']:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    # Restore W&B run
    wandb_run_id = checkpoint.get('wandb_run_id')
    if wandb_run_id and os.path.exists(WAND_ID_PATH):
        wandb.init(project=CONFIG['project_name'], id=wandb_run_id, resume="must")
        print(f"  ✓ Resumed W&B run: {wandb_run_id}")
    else:
        wandb.init(project=CONFIG['project_name'], name=CONFIG['run_name'], config=CONFIG)
        wandb_run_id = wandb.run.id
else:
    start_epoch = 0
    history = {k: [] for k in ['train_loss', 'val_loss', 'val_auc_joint', 'val_auc_audio',
                                'val_auc_video', 'lr', 'epoch_time']}
    best_val_auc = 0.0
    patience_counter = 0
    optimizer = make_optimizer(model, freeze=True)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    wandb.init(project=CONFIG['project_name'], name=CONFIG['run_name'], config=CONFIG)
    wandb_run_id = wandb.run.id
    wandb.watch(model, log="parameters", log_freq=100)

print(f"\nStarting from epoch {start_epoch+1}/{CONFIG['epochs']}")
print(f"Best AUC so far: {best_val_auc:.3f}, Patience: {patience_counter}/{CONFIG['patience']}")

# Training loop
import time

for epoch in range(start_epoch, CONFIG['epochs']):
    epoch_start = time.time()

    # Phase transition check
    if epoch == CONFIG['freeze_epochs']:
        print(f"\n>>> Phase transition at epoch {epoch+1}")
        optimizer = make_optimizer(model, freeze=False)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    # TRAINING
    model.train()
    train_loss = 0.0

    for batch in train_loader:
        v = batch['video'].to(device)
        a = batch['audio'].to(device)
        labels = batch['labels'].to(device)

        a_lab, v_lab = labels[:, 0:1], labels[:, 1:2]
        j_lab = ((a_lab == 1) & (v_lab == 1)).float()

        optimizer.zero_grad()
        out = model(v, a)

        loss = (criterion(out['audio_pred'], a_lab) +
               criterion(out['video_pred'], v_lab) +
               2.0 * criterion_hard(out['joint_pred'], j_lab))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # VALIDATION
    model.eval()
    val_loss = 0.0
    all_preds = {'audio': [], 'video': [], 'joint': []}
    all_labels = {'audio': [], 'video': [], 'joint': []}

    with torch.no_grad():
        for batch in val_loader:
            v = batch['video'].to(device)
            a = batch['audio'].to(device)
            labels = batch['labels'].to(device)

            a_lab, v_lab = labels[:, 0:1], labels[:, 1:2]
            j_lab = ((a_lab == 1) & (v_lab == 1)).float()

            out = model(v, a)
            val_loss += (criterion_hard(out['audio_pred'], a_lab) +
                        criterion_hard(out['video_pred'], v_lab) +
                        2.0 * criterion_hard(out['joint_pred'], j_lab)).item()

            for k in ['audio', 'video', 'joint']:
                all_preds[k].extend(out[f'{k}_pred'].cpu().numpy())
                all_labels[k].extend(locals()[f'{k}_lab'].cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)

    # Calculate AUCs
    def auc(y_true, y_pred):
        y_t, y_p = np.array(y_true).flatten(), np.array(y_pred).flatten()
        return roc_auc_score(y_t, y_p) if len(np.unique(y_t)) > 1 else 0.5

    val_auc_audio = auc(all_labels['audio'], all_preds['audio'])
    val_auc_video = auc(all_labels['video'], all_preds['video'])
    val_auc_joint = auc(all_labels['joint'], all_preds['joint'])

    # Update history
    epoch_time = time.time() - epoch_start
    current_lr = optimizer.param_groups[0]['lr'] if isinstance(optimizer.param_groups[0]['lr'], float) else optimizer.param_groups[0]['lr']

    for k, v in zip(history.keys(), [avg_train_loss, avg_val_loss, val_auc_joint, val_auc_audio,
                                      val_auc_video, current_lr, epoch_time]):
        history[k].append(v)

    # Log to W&B
    log_dict = {
        "epoch": epoch, "phase": "frozen" if epoch < CONFIG['freeze_epochs'] else "finetune",
        "train/loss": avg_train_loss, "val/loss": avg_val_loss,
        "val/auc_joint": val_auc_joint, "val/auc_audio": val_auc_audio, "val/auc_video": val_auc_video,
        "val/auc_gap": abs(val_auc_audio - val_auc_video),
        "train/loss_ratio": avg_train_loss / (avg_val_loss + 1e-8),
        "epoch_time": epoch_time, "learning_rate": current_lr
    }
    wandb.log(log_dict)

    # Console output
    phase = "[F]" if epoch < CONFIG['freeze_epochs'] else "[U]"
    print(f"Ep {epoch+1:2d}/{CONFIG['epochs']} {phase} | "
          f"Loss: T{avg_train_loss:.3f}/V{avg_val_loss:.3f} | "
          f"AUC: J{val_auc_joint:.2f} A{val_auc_audio:.2f} V{val_auc_video:.2f} | "
          f"Time: {epoch_time:.1f}s")

    # Scheduler step
    scheduler.step(avg_val_loss)

    # Checkpoint logic
    is_best = val_auc_joint > best_val_auc
    if is_best:
        best_val_auc = val_auc_joint
        patience_counter = 0
    else:
        patience_counter += 1

    # Save checkpoint every epoch (for resumption)
    if (epoch + 1) % CONFIG['checkpoint_freq'] == 0:
        save_checkpoint(epoch, model, optimizer, scheduler, history,
                       best_val_auc, patience_counter, wandb_run_id, is_best)

    # Early stopping
    if patience_counter >= CONFIG['patience']:
        print(f"\n⏹️ Early stopping at epoch {epoch+1} (best AUC: {best_val_auc:.3f})")
        break

print("\n✅ Training complete")

# Final save
save_checkpoint(epoch, model, optimizer, scheduler, history,
               best_val_auc, patience_counter, wandb_run_id, is_best=False)

# =============================================================================
# SECTION 6: FINAL EVALUATION
# =============================================================================

print("=" * 60)
print("SECTION 6: Final Evaluation")
print("=" * 60)

# Load best model
best_checkpoint = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(best_checkpoint['model_state_dict'])
model.eval()

print(f"Evaluating best model from epoch {best_checkpoint['epoch']+1}")

# Collect all predictions
all_results = []
for split_name, loader in [("train", train_loader), ("val", val_loader)]:
    with torch.no_grad():
        for batch in loader:
            out = model(batch['video'].to(device), batch['audio'].to(device))
            for i in range(len(batch['file'])):
                all_results.append({
                    'split': split_name,
                    'file': batch['file'][i],
                    'type': batch['type'][i],
                    'audio_gt': batch['labels'][i, 0].item(),
                    'video_gt': batch['labels'][i, 1].item(),
                    'audio_pred': out['audio_pred'][i].item(),
                    'video_pred': out['video_pred'][i].item(),
                    'joint_pred': out['joint_pred'][i].item()
                })

df = pd.DataFrame(all_results)

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Prediction space
ax1 = axes[0, 0]
colors = {'real': '#2ecc71', 'audio_modified': '#3498db',
          'visual_modified': '#e67e22', 'both_modified': '#e74c3c'}
for sp in ['train', 'val']:
    for t in df['type'].unique():
        sub = df[(df['type']==t) & (df['split']==sp)]
        ax1.scatter(sub['audio_pred'], sub['video_pred'],
                   c=colors.get(t), marker='s' if sp=='val' else 'o',
                   s=150 if sp=='val' else 80, alpha=0.8 if sp=='val' else 0.5,
                   edgecolors='black' if sp=='val' else 'none')
ax1.axhline(0.5, color='k', linestyle='--', alpha=0.3)
ax1.axvline(0.5, color='k', linestyle='--', alpha=0.3)
ax1.set_xlabel('Audio Prediction')
ax1.set_ylabel('Video Prediction')
ax1.set_title('Audio-Video Prediction Space')

# 2. Per-type accuracy
ax2 = axes[0, 1]
stats = []
for t in df['type'].unique():
    sub = df[df['type']==t]
    for m in ['audio', 'video', 'joint']:
        gt = sub[f'{m}_gt'] if m != 'joint' else ((sub['audio_gt']==0)|(sub['video_gt']==0)).astype(int)
        pred = (sub[f'{m}_pred'] > 0.5).astype(int)
        stats.append({'type': t, 'mod': m, 'acc': accuracy_score(gt, pred)})
df_stats = pd.DataFrame(stats).pivot(index='type', columns='mod', values='acc')
df_stats.plot(kind='bar', ax=ax2, color=['#3498db', '#e67e22', '#2ecc71'])
ax2.set_title('Per-Type Accuracy')
ax2.set_ylim(0, 1.1)

# 3. Prediction distribution
ax3 = axes[1, 0]
val_df = df[df['split']=='val']
ax3.hist(val_df['joint_pred'], bins=15, alpha=0.7, color='purple', edgecolor='black')
ax3.axvline(0.5, color='r', linestyle='--')
ax3.set_xlabel('Joint Prediction')
ax3.set_title('Validation Prediction Distribution')

# 4. Confusion matrix
ax4 = axes[1, 1]
y_true = ((val_df['audio_gt']==0)|(val_df['video_gt']==0)).astype(int)
y_pred = (val_df['joint_pred']>0.5).astype(int)
cm = confusion_matrix(y_true, y_pred)
im = ax4.imshow(cm, cmap='Blues')
ax4.set_xticks([0, 1])
ax4.set_yticks([0, 1])
ax4.set_xticklabels(['Pred Fake', 'Pred Real'])
ax4.set_yticklabels(['Actual Fake', 'Actual Real'])
for i in range(2):
    for j in range(2):
        ax4.text(j, i, cm[i,j], ha="center", va="center",
                color="white" if cm[i,j]>cm.max()/2 else "black", fontsize=14)
ax4.set_title('Confusion Matrix')

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'final_results.png'), dpi=150)
wandb.log({"final_analysis": wandb.Image(plt)})
plt.show()

# Summary
print(f"\nBest AUC: {best_checkpoint['best_val_auc']:.3f}")
wandb.summary.update({
    "best_epoch": best_checkpoint['epoch'],
    "best_auc": best_checkpoint['best_val_auc'],
    "total_epochs_trained": len(history['train_loss'])
})

print(f"\nView run: {wandb.run.url}")
wandb.finish()